In [ ]:
from pyspedas import mms_load_fgm, mms_load_fpi, \
    mms_load_hpca, mms_load_feeps, mms_load_edp, \
    mms_load_mec,mms_part_getspec
from pyspedas import tplot_names,get_data,store_data, \
    tsmooth,avg_data,time_datetime,time_double,subtract,\
    tinterpol,time_string
from pyspedas.projects.mms import mms_qcotrans 
# from pytplot import tplot,options
from pyspedas import tplot, options
from pyspedas import subtract_average
import numpy as np
import matplotlib.pyplot as plt
from functions import *
from scipy.constants import mu_0, m_p
import os
from itertools import combinations
import pandas as pd

## Read the selected interval (raw) and process gaps

In [ ]:
mms_id_list = ['1','2','3','4']  # list of MMS probes

Nsat = 4  # number of satellites

data_adjusted = pd.read_csv('./input/MMS_pristine_sw_intervals_selected.csv')


# print(data_adjusted)




In [ ]:
nintervals = len(data_adjusted)

print('Number of intervals: ', nintervals)

In [ ]:
dir_output = './output/Yaglom_analysis_all_selected_intervals/'
dir_figure = './figure/Yaglom_analysis_all_selected_intervals/'

if not os.path.exists(dir_output):
    os.makedirs(dir_output)

if not os.path.exists(dir_figure):
    os.makedirs(dir_figure)



good_intervals = []


for iinterval in range(nintervals):
    start_time = data_adjusted['start'][iinterval]
    end_time = data_adjusted['end'][iinterval]

    print('Processing interval ', iinterval+1, '/', nintervals, ':',  start_time, ' -- ', end_time)


    trange = [start_time, end_time]

    trange0_str = trange[0].replace(':', '-').replace('/', '-')
    trange1_str = trange[1].replace(':', '-').replace('/', '-')


    # do pre-processing for the current interval, including loading data, checking time steps, interpolating data, etc.-------
    # read orbit data for all MMS probes
    mms_load_mec(trange=trange, probe=mms_id_list, time_clip=True)

    mms_mec_r_gse = []
    for mms_id in mms_id_list:
        mms_mec_r_gse.append(get_data(f'mms{mms_id}_mec_r_gse', units=True))  # dt=True


    # get the coordinates of the four MMS probes in GSE
    t_mec_mms = []
    coord_mms = []
    coord_mms_RE = []
    for i, mms_id in enumerate(mms_id_list):
        t_mec_mms.append(mms_mec_r_gse[i].times)
        coord_mms.append(mms_mec_r_gse[i].y)
        coord_mms_RE.append(coord_mms[i] / Re)  # convert to RE



    # get velocity of the four MMS probes
    mms_mec_v_gse = []
    for mms_id in mms_id_list:
        mms_mec_v_gse.append(get_data(f'mms{mms_id}_mec_v_gse', units=True))  # dt=True


    v_mms = []
    for i, mms_id in enumerate(mms_id_list):
        v_mms.append(mms_mec_v_gse[i].y)  # velocity in GSE coordinates


    # # plot velocity of the four MMS probes
    # fig, subs = plt.subplots(4,1, figsize=(10, 10))
    # for i, mms_id in enumerate(mms_id_list):
    #     sub = subs[i]
    #     sub.plot(t_mec_mms[i], v_mms[i][:,0], label=r'$V_x$')
    #     sub.plot(t_mec_mms[i], v_mms[i][:,1], label=r'$V_y$')
    #     sub.plot(t_mec_mms[i], v_mms[i][:,2], label=r'$V_z$')
    #     sub.set_xlabel('Time (s)')
    #     sub.set_ylabel('Velocity (km/s)')
    #     sub.set_title(f'MMS{mms_id} Velocity in GSE')
    #     sub.legend()

    # fig.suptitle('MMS1-4 Velocity in GSE: ' + trange[0] + ' -- ' + trange[1])
    # plt.show()
    # plt.close(fig)




    # judge whether the four time series are identical
    allclose_mec_12 = np.allclose(t_mec_mms[0], t_mec_mms[1])
    allclose_mec_13 = np.allclose(t_mec_mms[0], t_mec_mms[2])
    allclose_mec_14 = np.allclose(t_mec_mms[0], t_mec_mms[3])
    if allclose_mec_12 and allclose_mec_13 and allclose_mec_14:
        print("All four MMS probes have identical time series for MEC data.")
    else:
        print("The time series for MEC data are not identical across all MMS probes.")
        if not allclose_mec_12:
            print("MMS1 and MMS2 have different time series.")
        if not allclose_mec_13:
            print("MMS1 and MMS3 have different time series.")
        if not allclose_mec_14:
            print("MMS1 and MMS4 have different time series.")


        print('Skip this interval and continue to the next one......')
        continue



    t_mec_center = np.copy(t_mec_mms[0])  # use the time of MMS1 as the reference time for all satellites

    coord_center_mms = (coord_mms[0] + coord_mms[1] + coord_mms[2] + coord_mms[3]) / Nsat


    coord_idff_mms = []
    for i in range(Nsat):
        coord_idff_mms.append(coord_mms[i] - coord_center_mms)






    # read the DIS data for all MMS probes
    fpi_data = mms_load_fpi(trange=trange, probe=mms_id_list, data_rate='fast', time_clip=True)
    fgm_data = mms_load_fgm(trange=trange, probe=mms_id_list, data_rate='srvy', time_clip=True)
    mec_data = mms_load_mec(trange=trange, probe=mms_id_list, time_clip=True)


    # despun ion velocity
    for mms_id in mms_id_list:
        subtract(f'mms{mms_id}_dis_bulkv_gse_fast', f'mms{mms_id}_dis_bulkv_spintone_gse_fast')



    
    # check time steps----------------------------------------------------------------------
    # get time steps for the four MMS probes - bulkv
    t_mms_dis_bulkv_gse_fast = []
    for mms_id in mms_id_list:
        t_mms_dis_bulkv_gse_fast.append(get_data(f'mms{mms_id}_dis_bulkv_gse_fast', units=True).times)


    dt_mms_dis_bulkv_gse_fast = []
    for i in range(Nsat):
        dt_mms_dis_bulkv_gse_fast.append(np.diff(t_mms_dis_bulkv_gse_fast[i]))


    dt_ave_mms_dis_bulkv_gse_fast = []
    dt_std_mms_dis_bulkv_gse_fast = []
    dt_std_norm_mms_dis_bulkv_gse_fast = []
    for i in range(Nsat):
        dt_ave_mms_dis_bulkv_gse_fast.append(np.nanmean(dt_mms_dis_bulkv_gse_fast[i]))
        dt_std_mms_dis_bulkv_gse_fast.append(np.nanstd(dt_mms_dis_bulkv_gse_fast[i]))
        dt_std_norm_mms_dis_bulkv_gse_fast.append(dt_std_mms_dis_bulkv_gse_fast[i] / dt_ave_mms_dis_bulkv_gse_fast[i])

    for i, mms_id in enumerate(mms_id_list):
        print(f'dt_ave_mms_dis_bulkv_gse_fast MMS{mms_id} = {dt_ave_mms_dis_bulkv_gse_fast[i]:.4f} s')


    for i, mms_id in enumerate(mms_id_list):
        print(f"variation(dt)/average(dt) MMS{mms_id}_dis_bulkv_gse_fast = {dt_std_norm_mms_dis_bulkv_gse_fast[i]:.4f}")



    # get time steps for the four MMS probes - number density
    t_mms_dis_ndens_fast = []
    for mms_id in mms_id_list:
        t_mms_dis_ndens_fast.append(get_data(f'mms{mms_id}_dis_numberdensity_fast', units=True).times)

    dt_mms_dis_ndens_fast = []
    for i in range(Nsat):
        dt_mms_dis_ndens_fast.append(np.diff(t_mms_dis_ndens_fast[i]))

    dt_ave_mms_dis_ndens_fast = []
    dt_std_mms_dis_ndens_fast = []
    dt_std_norm_mms_dis_ndens_fast = []
    for i in range(Nsat):
        dt_ave_mms_dis_ndens_fast.append(np.nanmean(dt_mms_dis_ndens_fast[i]))
        dt_std_mms_dis_ndens_fast.append(np.nanstd(dt_mms_dis_ndens_fast[i]))
        dt_std_norm_mms_dis_ndens_fast.append(dt_std_mms_dis_ndens_fast[i] / dt_ave_mms_dis_ndens_fast[i])

    for i, mms_id in enumerate(mms_id_list):
        print(f'dt_ave_mms_dis_numberdensity_fast MMS{mms_id} = {dt_ave_mms_dis_ndens_fast[i]:.4f} s')

    for i, mms_id in enumerate(mms_id_list):
        print(f"variation(dt)/average(dt) MMS{mms_id}_dis_numberdensity_fast = {dt_std_norm_mms_dis_ndens_fast[i]:.4f}")



    # check if the time series of bulkv and number density are identical
    allcose_bulkv_ndens_1 = np.allclose(t_mms_dis_bulkv_gse_fast[0], t_mms_dis_ndens_fast[0])
    allcose_bulkv_ndens_2 = np.allclose(t_mms_dis_bulkv_gse_fast[1], t_mms_dis_ndens_fast[1])
    allcose_bulkv_ndens_3 = np.allclose(t_mms_dis_bulkv_gse_fast[2], t_mms_dis_ndens_fast[2])
    allcose_bulkv_ndens_4 = np.allclose(t_mms_dis_bulkv_gse_fast[3], t_mms_dis_ndens_fast[3])

    if allcose_bulkv_ndens_1 and allcose_bulkv_ndens_2 and allcose_bulkv_ndens_3 and allcose_bulkv_ndens_4:
        print("All four MMS probes have identical time series for dis_bulkv_gse_fast and dis_numberdensity_fast data.")
    else:
        print("The time series for dis_bulkv_gse_fast and dis_numberdensity_fast data are not identical across all MMS probes.")
        if not allcose_bulkv_ndens_1:
            print("MMS1 has different time series.")
        if not allcose_bulkv_ndens_2:
            print("MMS2 has different time series.")
        if not allcose_bulkv_ndens_3:
            print("MMS3 has different time series.")
        if not allcose_bulkv_ndens_4:
            print("MMS4 has different time series.")

        print("Skip this interval due to to non-identical time series for bulkv and ndens.")
        continue



    # check time steps for magnetic field
    t_mms_fgm_b_gse_srvy_l2 = []
    for mms_id in mms_id_list:
        t_mms_fgm_b_gse_srvy_l2.append(get_data(f'mms{mms_id}_fgm_b_gse_srvy_l2', units=True).times)

    dt_mms_fgm_b_gse_srvy_l2 = []
    for i in range(Nsat):
        dt_mms_fgm_b_gse_srvy_l2.append(np.diff(t_mms_fgm_b_gse_srvy_l2[i]))

    dt_ave_mms_fgm_b_gse_srvy_l2 = []
    dt_std_mms_fgm_b_gse_srvy_l2 = []
    dt_std_norm_mms_fgm_b_gse_srvy_l2 = []
    for i in range(Nsat):
        dt_ave_mms_fgm_b_gse_srvy_l2.append(np.nanmean(dt_mms_fgm_b_gse_srvy_l2[i]))
        dt_std_mms_fgm_b_gse_srvy_l2.append(np.nanstd(dt_mms_fgm_b_gse_srvy_l2[i]))
        dt_std_norm_mms_fgm_b_gse_srvy_l2.append(dt_std_mms_fgm_b_gse_srvy_l2[i] / dt_ave_mms_fgm_b_gse_srvy_l2[i])



    # the magnetic field data have similar average time steps, but the variations can be large.
    for i, mms_id in enumerate(mms_id_list):
        print(f'dt_ave_mms_fgm_b_gse_srvy_l2 MMS{mms_id} = {dt_ave_mms_fgm_b_gse_srvy_l2[i]:.4f} s')
    for i, mms_id in enumerate(mms_id_list):
        print(f"variation(dt)/average(dt) MMS{mms_id}_fgm_b_gse_srvy_l2 = {dt_std_norm_mms_fgm_b_gse_srvy_l2[i]:.4f}")
    #---------------------------------------------------------------------
    



    # interplolate all fields to the MMS1 DIS
    tinterpol(['mms?_dis_numberdensity_fast','mms?_dis_bulkv_gse_fast','mms?_fgm_b_gse_srvy_l2',
            'mms?_mec_r_gse'], 
            'mms1_dis_numberdensity_fast', suffix='_interp')


    # tplot(['mms1_dis_numberdensity_fast', 'mms1_dis_numberdensity_fast_interp',
    #        'mms4_fgm_b_gse_srvy_l2', 'mms4_fgm_b_gse_srvy_l2_interp'])  # plot the interpolated data


    # subtract average and get v1
    for mms_id in mms_id_list:
        # subtract average from velocity
        subtract_average(f'mms{mms_id}_dis_bulkv_gse_fast_interp', f'mms{mms_id}_dis_bulkv1_gse_fast_interp')

        # print average velocity
        v_bulkv = get_data(f'mms{mms_id}_dis_bulkv_gse_fast_interp', units=False).y
        v_bulkv_ave = np.nanmean(v_bulkv, axis=0)
        print(f'Average velocity for MMS{mms_id}: {v_bulkv_ave} km/s')







    # the plasma data availability is limited, we need to further cut the time range of available data
    # method: scan the whole interpolated arrays, mark the range of available data
    # The indicator is very large time steps:
    # get the data 
    t_arr = get_data('mms1_dis_numberdensity_fast_interp', units=True).times
    # b_gse_interp = get_data('mms1_fgm_b_gse_srvy_l2_interp', units=False).y

    dt_arr = np.diff(t_arr)

    dt_mean = np.nanmean(dt_arr)

    markes_gap = []
    for i in range(len(dt_arr)):
        if dt_arr[i] > 5 * dt_mean:  # if the time step is larger than 5 times the average time step, we consider it as a gap
            print(f"Large time gap detected at index {i}: dt = {dt_arr[i]:.4f} s, mean dt = {dt_mean:.4f} s")
            print(f"Marking the gap between {time_datetime(t_arr[i])} and {time_datetime(t_arr[i+1])}")
            # we can further mark the data in this gap as NaN or remove it from the analysis
            # for simplicity, we will just print the information about the gap here
            markes_gap.append((t_arr[i], t_arr[i+1]))


    

    ngaps = len(markes_gap)
    ngood_intervals = ngaps + 1

    if ngood_intervals == 1:
        good_intervals.append((time_string(t_arr[0]), time_string(t_arr[-1])))
    else:
        good_intervals.append((time_string(t_arr[0]), time_string(markes_gap[0][0])))

        for i in range(ngaps-1):
            good_intervals.append((time_string(markes_gap[i][1]), time_string(markes_gap[i+1][0])))

        good_intervals.append((time_string(markes_gap[-1][1]), time_string(t_arr[-1])))
    





    fig, subs = plt.subplots(3,1,figsize=(10, 10))
    sub = subs[0]
    sub.plot(t_arr, get_data('mms1_dis_numberdensity_fast_interp', units=False).y)
    sub.set_ylabel(r'$N$',fontsize=12)

    sub = subs[1]
    sub.plot(t_arr, get_data('mms1_dis_bulkv1_gse_fast_interp', units=False).y[:,0], label=r'$v_x$')
    sub.plot(t_arr, get_data('mms1_dis_bulkv1_gse_fast_interp', units=False).y[:,1], label=r'$v_y$')
    sub.plot(t_arr, get_data('mms1_dis_bulkv1_gse_fast_interp', units=False).y[:,2], label=r'$v_z$')
    sub.set_ylabel(r'$v$ (km/s)',fontsize=12)
    sub.legend()

    sub = subs[2]
    bx = get_data('mms1_fgm_b_gse_srvy_l2_interp', units=False).y[:,0]
    by = get_data('mms1_fgm_b_gse_srvy_l2_interp', units=False).y[:,1]
    bz = get_data('mms1_fgm_b_gse_srvy_l2_interp', units=False).y[:,2]
    babs = np.sqrt(bx**2 + by**2 + bz**2)
    sub.plot(t_arr, get_data('mms1_fgm_b_gse_srvy_l2_interp', units=False).y[:,0], label=r'$B_x$')
    sub.plot(t_arr, get_data('mms1_fgm_b_gse_srvy_l2_interp', units=False).y[:,1], label=r'$B_y$')
    sub.plot(t_arr, get_data('mms1_fgm_b_gse_srvy_l2_interp', units=False).y[:,2], label=r'$B_z$')
    sub.plot(t_arr, babs, label=r'$|B|$',color='k')

    sub.set_ylabel(r'$B$ (nT)',fontsize=12)
    sub.legend()

    for sub in subs:
        if ngood_intervals>1:
            sub.axvspan(t_arr[0], markes_gap[0][0],color='yellow', alpha=0.3)

            for i in range(ngaps-1):
                sub.axvspan(markes_gap[i][1], markes_gap[i+1][0],color='yellow', alpha=0.3)

            sub.axvspan(markes_gap[-1][1], t_arr[-1],color='yellow', alpha=0.3)


    fig.savefig(dir_figure + trange0_str + '_interpolated_fields_marked_gaps.png')
    plt.close(fig)


    # tplot(['mms1_dis_numberdensity_fast_interp','mms1_dis_bulkv1_gse_fast_interp',
    #        'mms1_fgm_b_gse_srvy_l2_interp'], display=False, 
    #        save_png= dir_figure + trange0_str + '_interpolated_fields.png')


    continue

In [ ]:
# after marking the gaps, we can further cut the time range of available data and save the good intervals to a csv file
print(len(good_intervals))


# special note: I have mannually removed the interval starting from
# 2023-04-23 04-46-35.490027
# because MMS4 does not have data for this interval.


# save good intervals to a csv file
import csv
with open(dir_output + 'good_intervals_with_data_coverage.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Start Time', 'End Time'])
    for interval in good_intervals:
        writer.writerow(interval)

## Calculate $Y^\pm$ for each individual event

In [ ]:
# plot the good intervals on the original data to visually check the coverage
dir_output = './output/Yaglom_analysis_all_selected_intervals/'
dir_figure = './figure/Yaglom_analysis_all_selected_intervals/'

if not os.path.exists(dir_output):
    os.makedirs(dir_output)

if not os.path.exists(dir_figure):
    os.makedirs(dir_figure)



data_good_intervals = pd.read_csv(dir_output + 'good_intervals_with_data_coverage.csv')


nintervals = len(data_good_intervals)

print('Total number of good intervals: ', nintervals)



# add some extra requirements for the good intervals:

T_interval_min = 3600 # minimum duration of the interval, e.g. 3600s

tau_max_cap = 3600 * 4  # maximum time increment for structure function, e.g. 3600s



for i in range(nintervals):
    print(f"Interval {i+1}: Start Time = {data_good_intervals.iloc[i, 0]}, End Time = {data_good_intervals.iloc[i, 1]}")

In [ ]:

for iinterval in range(64,nintervals):

    # if iinterval !=11: 
    #     continue



    start_time = data_good_intervals['Start Time'].iloc[iinterval]
    end_time = data_good_intervals['End Time'].iloc[iinterval]

    print('Processing interval ', iinterval+1, '/', nintervals, ':',  start_time, ' -- ', end_time)


    # if start_time == "2023-01-18 09:16:01.407352":
    #     print("Skip this interval due to known data issue.")
    #     continue
    


    # check the duration of the interval
    duration = time_double(end_time) - time_double(start_time)
    if duration < T_interval_min:
        print(f"Skip this interval due to short duration: {duration:.2f} seconds.")
        continue



    trange = [start_time, end_time]

    trange0_str = trange[0].replace(':', '-').replace('/', '-')
    trange1_str = trange[1].replace(':', '-').replace('/', '-')


    # do pre-processing for the current interval, including loading data, checking time steps, interpolating data, etc.-------
    # read orbit data for all MMS probes
    mms_load_mec(trange=trange, probe=mms_id_list, time_clip=True)

    mms_mec_r_gse = []
    for mms_id in mms_id_list:
        mms_mec_r_gse.append(get_data(f'mms{mms_id}_mec_r_gse', units=True))  # dt=True


    # get the coordinates of the four MMS probes in GSE
    t_mec_mms = []
    coord_mms = []
    coord_mms_RE = []
    for i, mms_id in enumerate(mms_id_list):
        t_mec_mms.append(mms_mec_r_gse[i].times)
        coord_mms.append(mms_mec_r_gse[i].y)
        coord_mms_RE.append(coord_mms[i] / Re)  # convert to RE



    # get velocity of the four MMS probes
    mms_mec_v_gse = []
    for mms_id in mms_id_list:
        mms_mec_v_gse.append(get_data(f'mms{mms_id}_mec_v_gse', units=True))  # dt=True


    v_mms = []
    for i, mms_id in enumerate(mms_id_list):
        v_mms.append(mms_mec_v_gse[i].y)  # velocity in GSE coordinates


    # plot velocity of the four MMS probes
    # fig, subs = plt.subplots(4,1, figsize=(10, 10))
    # for i, mms_id in enumerate(mms_id_list):
    #     sub = subs[i]
    #     sub.plot(t_mec_mms[i], v_mms[i][:,0], label=r'$V_x$')
    #     sub.plot(t_mec_mms[i], v_mms[i][:,1], label=r'$V_y$')
    #     sub.plot(t_mec_mms[i], v_mms[i][:,2], label=r'$V_z$')
    #     sub.set_xlabel('Time (s)')
    #     sub.set_ylabel('Velocity (km/s)')
    #     sub.set_title(f'MMS{mms_id} Velocity in GSE')
    #     sub.legend()

    # fig.suptitle('MMS1-4 Velocity in GSE: ' + trange[0] + ' -- ' + trange[1])
    # plt.show()
    # plt.close(fig)




    # judge whether the four time series are identical
    allclose_mec_12 = np.allclose(t_mec_mms[0], t_mec_mms[1])
    allclose_mec_13 = np.allclose(t_mec_mms[0], t_mec_mms[2])
    allclose_mec_14 = np.allclose(t_mec_mms[0], t_mec_mms[3])
    if allclose_mec_12 and allclose_mec_13 and allclose_mec_14:
        print("All four MMS probes have identical time series for MEC data.")
    else:
        print("The time series for MEC data are not identical across all MMS probes.")
        if not allclose_mec_12:
            print("MMS1 and MMS2 have different time series.")
        if not allclose_mec_13:
            print("MMS1 and MMS3 have different time series.")
        if not allclose_mec_14:
            print("MMS1 and MMS4 have different time series.")


        print('Skip this interval and continue to the next one......')
        continue



    t_mec_center = np.copy(t_mec_mms[0])  # use the time of MMS1 as the reference time for all satellites

    coord_center_mms = (coord_mms[0] + coord_mms[1] + coord_mms[2] + coord_mms[3]) / Nsat


    coord_idff_mms = []
    for i in range(Nsat):
        coord_idff_mms.append(coord_mms[i] - coord_center_mms)






    # read the DIS data for all MMS probes
    fpi_data = mms_load_fpi(trange=trange, probe=mms_id_list, data_rate='fast', time_clip=True)
    fgm_data = mms_load_fgm(trange=trange, probe=mms_id_list, data_rate='srvy', time_clip=True)
    mec_data = mms_load_mec(trange=trange, probe=mms_id_list, time_clip=True)


    # despun ion velocity
    for mms_id in mms_id_list:
        subtract(f'mms{mms_id}_dis_bulkv_gse_fast', f'mms{mms_id}_dis_bulkv_spintone_gse_fast')

    
    # check time steps----------------------------------------------------------------------
    # get time steps for the four MMS probes - bulkv
    t_mms_dis_bulkv_gse_fast = []
    for mms_id in mms_id_list:
        t_mms_dis_bulkv_gse_fast.append(get_data(f'mms{mms_id}_dis_bulkv_gse_fast', units=True).times)


    dt_mms_dis_bulkv_gse_fast = []
    for i in range(Nsat):
        dt_mms_dis_bulkv_gse_fast.append(np.diff(t_mms_dis_bulkv_gse_fast[i]))


    dt_ave_mms_dis_bulkv_gse_fast = []
    dt_std_mms_dis_bulkv_gse_fast = []
    dt_std_norm_mms_dis_bulkv_gse_fast = []
    for i in range(Nsat):
        dt_ave_mms_dis_bulkv_gse_fast.append(np.nanmean(dt_mms_dis_bulkv_gse_fast[i]))
        dt_std_mms_dis_bulkv_gse_fast.append(np.nanstd(dt_mms_dis_bulkv_gse_fast[i]))
        dt_std_norm_mms_dis_bulkv_gse_fast.append(dt_std_mms_dis_bulkv_gse_fast[i] / dt_ave_mms_dis_bulkv_gse_fast[i])

    for i, mms_id in enumerate(mms_id_list):
        print(f'dt_ave_mms_dis_bulkv_gse_fast MMS{mms_id} = {dt_ave_mms_dis_bulkv_gse_fast[i]:.4f} s')


    for i, mms_id in enumerate(mms_id_list):
        print(f"variation(dt)/average(dt) MMS{mms_id}_dis_bulkv_gse_fast = {dt_std_norm_mms_dis_bulkv_gse_fast[i]:.4f}")



    # get time steps for the four MMS probes - number density
    t_mms_dis_ndens_fast = []
    for mms_id in mms_id_list:
        t_mms_dis_ndens_fast.append(get_data(f'mms{mms_id}_dis_numberdensity_fast', units=True).times)

    dt_mms_dis_ndens_fast = []
    for i in range(Nsat):
        dt_mms_dis_ndens_fast.append(np.diff(t_mms_dis_ndens_fast[i]))

    dt_ave_mms_dis_ndens_fast = []
    dt_std_mms_dis_ndens_fast = []
    dt_std_norm_mms_dis_ndens_fast = []
    for i in range(Nsat):
        dt_ave_mms_dis_ndens_fast.append(np.nanmean(dt_mms_dis_ndens_fast[i]))
        dt_std_mms_dis_ndens_fast.append(np.nanstd(dt_mms_dis_ndens_fast[i]))
        dt_std_norm_mms_dis_ndens_fast.append(dt_std_mms_dis_ndens_fast[i] / dt_ave_mms_dis_ndens_fast[i])

    for i, mms_id in enumerate(mms_id_list):
        print(f'dt_ave_mms_dis_numberdensity_fast MMS{mms_id} = {dt_ave_mms_dis_ndens_fast[i]:.4f} s')

    for i, mms_id in enumerate(mms_id_list):
        print(f"variation(dt)/average(dt) MMS{mms_id}_dis_numberdensity_fast = {dt_std_norm_mms_dis_ndens_fast[i]:.4f}")



    # check if the time series of bulkv and number density are identical
    allcose_bulkv_ndens_1 = np.allclose(t_mms_dis_bulkv_gse_fast[0], t_mms_dis_ndens_fast[0])
    allcose_bulkv_ndens_2 = np.allclose(t_mms_dis_bulkv_gse_fast[1], t_mms_dis_ndens_fast[1])
    allcose_bulkv_ndens_3 = np.allclose(t_mms_dis_bulkv_gse_fast[2], t_mms_dis_ndens_fast[2])
    allcose_bulkv_ndens_4 = np.allclose(t_mms_dis_bulkv_gse_fast[3], t_mms_dis_ndens_fast[3])

    if allcose_bulkv_ndens_1 and allcose_bulkv_ndens_2 and allcose_bulkv_ndens_3 and allcose_bulkv_ndens_4:
        print("All four MMS probes have identical time series for dis_bulkv_gse_fast and dis_numberdensity_fast data.")
    else:
        print("The time series for dis_bulkv_gse_fast and dis_numberdensity_fast data are not identical across all MMS probes.")
        if not allcose_bulkv_ndens_1:
            print("MMS1 has different time series.")
        if not allcose_bulkv_ndens_2:
            print("MMS2 has different time series.")
        if not allcose_bulkv_ndens_3:
            print("MMS3 has different time series.")
        if not allcose_bulkv_ndens_4:
            print("MMS4 has different time series.")

        print("Skip this interval due to to non-identical time series for bulkv and ndens.")
        continue



    # check time steps for magnetic field
    t_mms_fgm_b_gse_srvy_l2 = []
    for mms_id in mms_id_list:
        t_mms_fgm_b_gse_srvy_l2.append(get_data(f'mms{mms_id}_fgm_b_gse_srvy_l2', units=True).times)

    dt_mms_fgm_b_gse_srvy_l2 = []
    for i in range(Nsat):
        dt_mms_fgm_b_gse_srvy_l2.append(np.diff(t_mms_fgm_b_gse_srvy_l2[i]))

    dt_ave_mms_fgm_b_gse_srvy_l2 = []
    dt_std_mms_fgm_b_gse_srvy_l2 = []
    dt_std_norm_mms_fgm_b_gse_srvy_l2 = []
    for i in range(Nsat):
        dt_ave_mms_fgm_b_gse_srvy_l2.append(np.nanmean(dt_mms_fgm_b_gse_srvy_l2[i]))
        dt_std_mms_fgm_b_gse_srvy_l2.append(np.nanstd(dt_mms_fgm_b_gse_srvy_l2[i]))
        dt_std_norm_mms_fgm_b_gse_srvy_l2.append(dt_std_mms_fgm_b_gse_srvy_l2[i] / dt_ave_mms_fgm_b_gse_srvy_l2[i])



    # the magnetic field data have similar average time steps, but the variations can be large.
    for i, mms_id in enumerate(mms_id_list):
        print(f'dt_ave_mms_fgm_b_gse_srvy_l2 MMS{mms_id} = {dt_ave_mms_fgm_b_gse_srvy_l2[i]:.4f} s')
    for i, mms_id in enumerate(mms_id_list):
        print(f"variation(dt)/average(dt) MMS{mms_id}_fgm_b_gse_srvy_l2 = {dt_std_norm_mms_fgm_b_gse_srvy_l2[i]:.4f}")
    #---------------------------------------------------------------------
    



    # interplolate all fields to the MMS1 DIS
    tinterpol(['mms?_dis_numberdensity_fast','mms?_dis_bulkv_gse_fast','mms?_fgm_b_gse_srvy_l2',
            'mms?_mec_r_gse'], 
            'mms1_dis_numberdensity_fast', suffix='_interp')


    # tplot(['mms1_dis_numberdensity_fast', 'mms1_dis_numberdensity_fast_interp',
    #        'mms4_fgm_b_gse_srvy_l2', 'mms4_fgm_b_gse_srvy_l2_interp'])  # plot the interpolated data


    # subtract average and get v1
    for mms_id in mms_id_list:
        # subtract average from velocity
        subtract_average(f'mms{mms_id}_dis_bulkv_gse_fast_interp', f'mms{mms_id}_dis_bulkv1_gse_fast_interp')

        # print average velocity
        v_bulkv = get_data(f'mms{mms_id}_dis_bulkv_gse_fast_interp', units=False).y
        v_bulkv_ave = np.nanmean(v_bulkv, axis=0)
        print(f'Average velocity for MMS{mms_id}: {v_bulkv_ave} km/s')





    tplot(['mms1_dis_numberdensity_fast_interp','mms1_dis_bulkv1_gse_fast_interp',
           'mms1_fgm_b_gse_srvy_l2_interp'], display=False, 
           save_png= dir_figure + trange0_str + '_interpolated_fields.png')




    mms_mec_r_gse = []
    for mms_id in mms_id_list:
        mms_mec_r_gse.append(get_data(f'mms{mms_id}_mec_r_gse_interp', units=False))  # dt=True

    # get the coordinates of the four MMS probes in GSE
    t_mec_mms = []
    coord_mms = []
    for i, mms_id in enumerate(mms_id_list):
        t_mec_mms.append(mms_mec_r_gse[i].times)
        coord_mms.append(mms_mec_r_gse[i].y)

    coord_center_mms = (coord_mms[0] + coord_mms[1] + coord_mms[2] + coord_mms[3]) / Nsat


    coord_diff_mms = []
    for i in range(Nsat):
        coord_diff_mms.append(coord_mms[i] - coord_center_mms)

    coord_diff_mms_abs = []
    for i in range(Nsat):
        coord_diff_mms_abs.append(np.sqrt(np.sum(coord_diff_mms[i]**2, axis=1)))

    norm_variation_coord_diff_mms = []
    for i in range(Nsat):
        norm_variation_coord_diff_mms.append((np.nanmax(coord_diff_mms_abs[i]) - np.nanmin(coord_diff_mms_abs[i])) / np.nanmean(coord_diff_mms_abs[i]))

    for i, mms_id in enumerate(mms_id_list):
        print(f"Normalized variation of distance from center for MMS{mms_id}: {norm_variation_coord_diff_mms[i]:.4f}")

    t_mec_center = np.copy(t_mec_mms[0])



    # plot the absolute distances of the four MMS satellites from the center of the tetrahedron
    fig, sub = plt.subplots(1,1)
    for i, mms_id in enumerate(mms_id_list):
        sub.plot(t_mec_center, coord_diff_mms_abs[i], label=f'MMS{mms_id}')
    sub.set_xlabel('Time (s)')
    sub.set_ylabel('Distance from Center (km)')
    sub.set_title('Distance of MMS Satellites from Center of Tetrahedron')
    sub.legend()
    fig.suptitle('MMS1-4 Relative Distances: ' + trange[0] + ' -- ' + trange[1])
    plt.tight_layout()
    # plt.show()
    fig.savefig(dir_figure + trange0_str + '_distance_from_center.png')
    plt.close(fig)


    # plot the relative positions of the four MMS satellites
    fig, subs = plt.subplots(1,3, figsize=(15, 5))

    sub = subs[0]
    # X-Y plane
    sub.plot(coord_diff_mms[0][:,0], coord_diff_mms[0][:,1], label='MMS1')
    sub.plot(coord_diff_mms[1][:,0], coord_diff_mms[1][:,1], label='MMS2')
    sub.plot(coord_diff_mms[2][:,0], coord_diff_mms[2][:,1], label='MMS3')
    sub.plot(coord_diff_mms[3][:,0], coord_diff_mms[3][:,1], label='MMS4')
    sub.set_xlabel('X (km) / GSE')
    sub.set_ylabel('Y (km) / GSE')
    sub.set_title('Relative Positions in X-Y Plane')
    sub.legend()

    sub = subs[1]
    # X-Z plane
    sub.plot(coord_diff_mms[0][:,0], coord_diff_mms[0][:,2], label='MMS1')
    sub.plot(coord_diff_mms[1][:,0], coord_diff_mms[1][:,2], label='MMS2')
    sub.plot(coord_diff_mms[2][:,0], coord_diff_mms[2][:,2], label='MMS3')
    sub.plot(coord_diff_mms[3][:,0], coord_diff_mms[3][:,2], label='MMS4')
    sub.set_xlabel('X (km) / GSE')
    sub.set_ylabel('Z (km) / GSE')
    sub.set_title('Relative Positions in X-Z Plane')

    sub = subs[2]
    # Y-Z plane
    sub.plot(coord_diff_mms[0][:,1], coord_diff_mms[0][:,2], label='MMS1')
    sub.plot(coord_diff_mms[1][:,1], coord_diff_mms[1][:,2], label='MMS2')
    sub.plot(coord_diff_mms[2][:,1], coord_diff_mms[2][:,2], label='MMS3')
    sub.plot(coord_diff_mms[3][:,1], coord_diff_mms[3][:,2], label='MMS4')
    sub.set_xlabel('Y (km) / GSE')
    sub.set_ylabel('Z (km) / GSE')
    sub.set_title('Relative Positions in Y-Z Plane')
    fig.suptitle('Relative Positions of MMS1-4: ' + trange[0] + ' -- ' + trange[1])
    plt.tight_layout()
    # plt.show()
    fig.savefig(dir_figure + trange0_str + '_relative_positions.png')
    plt.close(fig)


    # plot the trajectory of the four MMS satellites in all three planes
    fig,subs = plt.subplots(1,3, figsize=(15, 5))

    sub = subs[0]
    sub.plot(coord_mms[0][:,0],coord_mms[0][:,1],label='MMS1')
    sub.plot(coord_mms[1][:,0],coord_mms[1][:,1],label='MMS2')
    sub.plot(coord_mms[2][:,0],coord_mms[2][:,1],label='MMS3')
    sub.plot(coord_mms[3][:,0],coord_mms[3][:,1],label='MMS4')
    sub.legend()
    sub.set_xlabel('X (km) / GSE')
    sub.set_ylabel('Y (km) / GSE')

    sub = subs[1]
    sub.plot(coord_mms[0][:,0],coord_mms[0][:,2],label='MMS1')
    sub.plot(coord_mms[1][:,0],coord_mms[1][:,2],label='MMS2')
    sub.plot(coord_mms[2][:,0],coord_mms[2][:,2],label='MMS3')
    sub.plot(coord_mms[3][:,0],coord_mms[3][:,2],label='MMS4')
    sub.set_xlabel('X (km) / GSE')
    sub.set_ylabel('Z (km) / GSE')

    sub = subs[2]
    sub.plot(coord_mms[0][:,1],coord_mms[0][:,2],label='MMS1')
    sub.plot(coord_mms[1][:,1],coord_mms[1][:,2],label='MMS2')
    sub.plot(coord_mms[2][:,1],coord_mms[2][:,2],label='MMS3')
    sub.plot(coord_mms[3][:,1],coord_mms[3][:,2],label='MMS4')
    sub.set_xlabel('Y (km) / GSE')
    sub.set_ylabel('Z (km) / GSE')

    fig.suptitle('MMS1-4 Trajectory: '+ trange[0] + ' -- ' + trange[1])
    # plt.show()
    fig.savefig(dir_figure + trange0_str + '_trajectory.png')
    plt.close(fig)







    # calculate average solar wind speed and density
    ndens_avg = []
    bulkv_avg = []
    B_avg = []

    for mms_id in mms_id_list:
        ndens = get_data(f'mms{mms_id}_dis_numberdensity_fast_interp')
        bulkv = get_data(f'mms{mms_id}_dis_bulkv_gse_fast_interp')
        B = get_data(f'mms{mms_id}_fgm_b_gse_srvy_l2_interp')

        t = ndens.times  # use the time of the number density data


        # take the time range 
        t_range = np.where((t >= time_double(trange[0])) & (t <= time_double(trange[1])))[0]

        ndens_avg.append(np.nanmean(ndens.y[t_range]))
        bulkv_avg.append(np.nanmean(bulkv.y[t_range], axis=0))  # average over the time range
        B_avg.append(np.nanmean(B.y[t_range], axis=0))  # average over the time range


        # get average coord_diff_mms in time



    # for i, mms_id in enumerate(mms_id_list):
    #     print(f'MMS{mms_id} average number density: {ndens_avg[i]:.4f} cm^-3')
    #     print(f'MMS{mms_id} average bulk velocity: {bulkv_avg[i][0]:.4f} km/s, {bulkv_avg[i][1]:.4f} km/s, {bulkv_avg[i][2]:.4f} km/s')
    #     print(f'MMS{mms_id} average magnetic field: {B_avg[i][0]:.4f} nT, {B_avg[i][1]:.4f} nT, {B_avg[i][2]:.4f} nT')



    ndens_avg = np.mean(np.array(ndens_avg))
    bulkv_avg = np.mean(np.array(bulkv_avg), axis=0)  # average over the four MMS probes
    B_avg = np.mean(np.array(B_avg), axis=0)  # average over the four MMS

    print(f'Average number density: {ndens_avg:.4f} cm^-3')
    print(f'Average bulk velocity: {bulkv_avg[0]:.4f} km/s, {bulkv_avg[1]:.4f} km/s, {bulkv_avg[2]:.4f} km/s')
    print(f'Average magnetic field: {B_avg[0]:.4f} nT, {B_avg[1]:.4f} nT, {B_avg[2]:.4f} nT')


    coord_diff_mms_avg = []
    for i in range(Nsat):
        # print(coord_diff_mms[i].shape)
        coord_diff_mms_avg.append(np.nanmean(coord_diff_mms[i], axis=0))  # average over the time range



    coord_diff_mms_rms = []
    for i in range(Nsat):
        # coord_diff_mms[i] has dimension (N_time, 3)
        # I want to calculate the RMS variation of the vector over time, which is sqrt(mean((x-<average>)^2 + (y-<average>)^2 + (z-<average>)^2))
        coord_diff_mms_rms.append(np.sqrt(np.nanmean(np.sum((coord_diff_mms[i]-coord_diff_mms_avg[i])**2, axis=1), axis=0)))  # RMS over the time range


    coord_diff_mms_range = []
    for i in range(Nsat):
        coord_diff_mms_range.append(np.nanmax(coord_diff_mms[i], axis=0) - np.nanmin(coord_diff_mms[i], axis=0))  # range over the time range

    polarity_B0 = np.sign(B_avg[0])



    # calculate z+ and z- for each MMS probe 
    zp_list = []
    zm_list = []
    for mms_id in mms_id_list:
        V = get_data(f'mms{mms_id}_dis_bulkv_gse_fast_interp').y
        B = get_data(f'mms{mms_id}_fgm_b_gse_srvy_l2_interp').y[:,0:3]

        t = get_data(f'mms{mms_id}_dis_bulkv_gse_fast_interp').times  # use the time of the bulk velocity data

        t_range = np.where((t >= time_double(trange[0])) & (t <= time_double(trange[1])))[0]

        V = V[t_range]
        B = B[t_range]

        v1 = np.zeros(V.shape)
        b1 = np.zeros(B.shape)
        v1[:,0] = V[:,0] - bulkv_avg[0]  # subtract the average bulk velocity
        v1[:,1] = V[:,1] - bulkv_avg[1]  # subtract the average bulk velocity
        v1[:,2] = V[:,2] - bulkv_avg[2]  # subtract the average bulk velocity

        b1[:,0] = B[:,0] - B_avg[0]  # subtract the average magnetic field
        b1[:,1] = B[:,1] - B_avg[1]  # subtract the average magnetic field
        b1[:,2] = B[:,2] - B_avg[2]  # subtract the average magnetic field

        va1 = b1 * 1e-9 /np.sqrt(ndens_avg * 1e6 * m_p * mu_0) * 1e-3 # km/s

        if polarity_B0 > 0: # sunward magnetic field: outward wave is z+ = u + va
            zp = v1 + va1
            zm = v1 - va1
        else: # anti-sunward magnetic field: outward wave is z+ = u - va
            zp = v1 - va1
            zm = v1 + va1

        
        zp_list.append(zp)
        zm_list.append(zm)


        store_data(f'mms{mms_id}_zp_gse', data={'x': t, 'y': zp})
        store_data(f'mms{mms_id}_zm_gse', data={'x': t, 'y': zm})

        # print(np.nanstd(zp),np.nanstd(zm))  # print the standard deviation of z+ and z- for each MMS probe


    # exit()

    t_arr = np.copy(t)


    tplot(['mms1_zp_gse', 'mms2_zp_gse', 'mms3_zp_gse', 'mms4_zp_gse'], display=False, 
        save_png= dir_figure + trange0_str + '_z_plus.png')
    tplot(['mms1_zm_gse', 'mms2_zm_gse', 'mms3_zm_gse', 'mms4_zm_gse'], display=False, 
        save_png= dir_figure + trange0_str + '_z_minus.png')




    # Begin evaluate the structure function-----------------------
    # Step 1: give series of tau: time increment 
    T_tot = t_arr[-1] - t_arr[0]  # total time duration
    dt_arr = t_arr - t_arr[0]

    tau_min = 60 # minimum time increment, e.g. 60s

    # set the maximum time increment for structure function calculation based on the duration of the interval
    if T_tot/2 < tau_max_cap:
        tau_max = T_tot / 2
        print(f"Set tau_max to {tau_max:.2f} seconds for this interval.")
    else:
        tau_max = tau_max_cap
        print(f"Set tau_max to {tau_max:.2f} seconds for this interval.")


    ind_tau_min = np.where(dt_arr >= tau_min)[0][0]  # index of the minimum time increment
    ind_tau_max = np.where(dt_arr <= tau_max)[0][-1]  # index of the maximum time increment

    tau_arr = dt_arr[ind_tau_min:ind_tau_max+1]  # time increments from tau_min to tau_max


    ntau = len(tau_arr)  # number of time increments

    dX_center = np.zeros([ntau,3])  # distance-increment of the center of the tetrahedron in GSE coordinates



    Yp_arr = np.zeros((ntau, Nsat, Nsat, 3))  # Yp_arr[tau, mms_id1, mms_id2, 3] for (dz+^2) * dz- structure function
    Ym_arr = np.zeros((ntau, Nsat, Nsat, 3))  # Ym_arr[tau, mms_id1, mms_id2, 3] for (dz-^2) * dz+ structure function




    # Step 2: for each tau, we will scan the whole time series and calculate the structure function
    for itau in range(len(tau_arr)):
        progress_bar(itau+1,len(tau_arr), bar_length=50)  # print progress bar
        tau = tau_arr[itau]
        dindex = int(tau/dt_arr[1])
        

        dX_center[itau,:] = - bulkv_avg * tau

        # determine the start and end indices for the array
        start_index = 0
        end_index = len(t_arr) - dindex - 1

        if end_index <= start_index:
            print(f"Error: end_index ({end_index}) is not greater than start_index ({start_index}) for tau = {tau}.")
            continue


        if end_index + dindex >= len(t_arr):
            print(f"Error: end_index + dindex ({end_index + dindex}) exceeds the length of t_arr ({len(t_arr)}) for tau = {tau}.")
            continue


        for isat in range(Nsat):
            for jsat in range(Nsat):
                # between MMS(isat+1) and MMS(jsat+1)
                Yp_tmp = np.zeros(3)
                Ym_tmp = np.zeros(3)

                n_int = 0
                for i in range(start_index, end_index + 1):
                    index0 = i 
                    index1 = i + dindex 

                    dzp = zp_list[jsat][index1,:] - zp_list[isat][index0,:]
                    dzm = zm_list[jsat][index1,:] - zm_list[isat][index0,:]

                    if np.isnan(dzp).any() or np.isnan(dzm).any():
                        continue


                    Yp_tmp += (dzp[0]*dzp[0] + dzp[1]*dzp[1] + dzp[2]*dzp[2]) * dzm
                    Ym_tmp += (dzm[0]*dzm[0] + dzm[1]*dzm[1] + dzm[2]*dzm[2]) * dzp

                    n_int += 1  # count the number of intervals

                Yp_arr[itau, isat, jsat, :] = Yp_tmp / n_int
                Ym_arr[itau, isat, jsat, :] = Ym_tmp / n_int


    file_output = dir_output + 'Yaglom_law_' + trange0_str + '.npz'
    
    print("Finished calculating the structure functions for all tau. Save data to .npz file......")
    np.savez(file_output, trange = trange, 
            tau_arr = tau_arr, ndens_avg = ndens_avg, bulkv_avg = bulkv_avg, 
            B_avg = B_avg, coord_diff_mms_avg = coord_diff_mms_avg, coord_diff_mms_rms = coord_diff_mms_rms,
            coord_diff_mms_range = coord_diff_mms_range,
            dX_center = dX_center, Yp_arr = Yp_arr, Ym_arr = Ym_arr)



In [ ]:
tplot(['mms4_dis_bulkv_gse_fast'])

In [ ]:


data = np.load(dir_output + 'Yaglom_law_2023-01-18 09-16-01.407352.npz')



print(data['coord_diff_mms_avg'])

print(data['coord_diff_mms_rms'])

print(data['coord_diff_mms_range'])


In [ ]:






# useless
for i in range(1):
    continue
    '''
    # t_window to remove spin tones
    t_window_de_spin = 20

    # dt_in_double = time_double('2018-1-1 00:00:20.000000') - time_double('2018-1-1 00:00:00')  # 20 seconds in double format

    # # seems to be in seconds, i.e. equal to t_window_de_spin
    # print(dt_in_double)


    # name for dis_density : 'mms1_dis_numberdensity_fast'
    mms_dis_ndens_fast_names = []
    for mms_id in mms_id_list:
        mms_dis_ndens_fast_names.append(f'mms{mms_id}_dis_numberdensity_fast')


    # name for velocity : 'mms1_dis_bulkv_gse_fast'

    # # this avg_data function cannot give identical time arrays because the time stamps of the four MMS probes are not exactly identical
    # mms_dis_ndens_fast_avg_names = avg_data(mms_dis_ndens_fast_names, trange=trange,res=t_window_de_spin,
    #                                   suffix='_avg')


    # average density data for all MMS probes
    mms_dis_ndens_fast_avg = []
    mms_dis_ndens_fast_avg_names = []
    for i, mms_id in enumerate(mms_id_list):
        avg_data_name = tvariable_smooth(mms_dis_ndens_fast_names[i],trange,t_window_de_spin,suffix='_avg')

        mms_dis_ndens_fast_avg.append(get_data(avg_data_name, units=True))  # dt=True
        mms_dis_ndens_fast_avg_names.append(avg_data_name)  # store the name of the averaged data

    # average velocity data for all MMS probes
    mms_dis_bulkv_gse_fast_avg = []
    mms_dis_bulkv_gse_fast_avg_names = []
    for i, mms_id in enumerate(mms_id_list):
        avg_data_name = tvariable_smooth(f'mms{mms_id}_dis_bulkv_gse_fast',trange,t_window_de_spin,suffix='_avg')

        mms_dis_bulkv_gse_fast_avg.append(get_data(avg_data_name, units=True))  # dt=True
        mms_dis_bulkv_gse_fast_avg_names.append(avg_data_name)  # store the name of the averaged data


    # average magnetic field data for all MMS probes
    mms_fgm_b_gse_srvy_l2_avg = []
    mms_fgm_b_gse_srvy_l2_avg_names = []
    for i, mms_id in enumerate(mms_id_list):
        avg_data_name = tvariable_smooth(f'mms{mms_id}_fgm_b_gse_srvy_l2',trange,t_window_de_spin,suffix='_avg')

        mms_fgm_b_gse_srvy_l2_avg.append(get_data(avg_data_name, units=True))  # dt=True
        mms_fgm_b_gse_srvy_l2_avg_names.append(avg_data_name)  # store the name of the averaged data



    # plot the averaged data
    tplot([mms_dis_ndens_fast_avg_names[3],mms_dis_bulkv_gse_fast_avg_names[3],
        mms_fgm_b_gse_srvy_l2_avg_names[3]], display=False,
        save_png = dir_figure + trange0_str + '_averaged_data.png')  



    mms_dis_numberdensity_fast = []
    for mms_id in mms_id_list:
        mms_dis_numberdensity_fast.append(get_data(f'mms{mms_id}_dis_numberdensity_fast', units=True))


    t_mms_dis_ndens_fast = []
    for i in range(Nsat):
        t_mms_dis_ndens_fast.append(mms_dis_numberdensity_fast[i].times)


    # print(t_mms1_dis_numberdensity_fast[2348:])
    # print(t_mms2_dis_numberdensity_fast[2348:])
    # print(t_mms3_dis_numberdensity_fast[2348:])
    # print(t_mms4_dis_numberdensity_fast[2348:])

    # calculate dt 
    dt_mms_dis_ndens_fast = []
    for i in range(Nsat):
        dt_mms_dis_ndens_fast.append(np.diff(t_mms_dis_ndens_fast[i]))

    dt_ave_mms_dis_ndens_fast = []
    dt_std_mms_dis_ndens_fast = []
    dt_std_norm_mms_dis_ndens_fast = []
    for i in range(Nsat):
        dt_ave_mms_dis_ndens_fast.append(np.nanmean(dt_mms_dis_ndens_fast[i]))
        dt_std_mms_dis_ndens_fast.append(np.nanstd(dt_mms_dis_ndens_fast[i]))
        dt_std_norm_mms_dis_ndens_fast.append(dt_std_mms_dis_ndens_fast[i] / dt_ave_mms_dis_ndens_fast[i])

    # # whether the dt_ave are identical for the four MMS probes  -- not exactly identical, but very close
    # if dt_ave_mms1_dis_numberdensity_fast == dt_ave_mms2_dis_numberdensity_fast and \
    #     dt_ave_mms1_dis_numberdensity_fast == dt_ave_mms3_dis_numberdensity_fast and \
    #     dt_ave_mms1_dis_numberdensity_fast == dt_ave_mms4_dis_numberdensity_fast:
    #     print("The average time steps for dis_numberdensity_fast data are identical across all MMS probes.")
    # else:
    #     print("The average time steps for dis_numberdensity_fast data are not identical across all MMS probes.")
    #     print(f"MMS1: {dt_ave_mms1_dis_numberdensity_fast:.4f} s")
    #     print(f"MMS2: {dt_ave_mms2_dis_numberdensity_fast:.4f} s")
    #     print(f"MMS3: {dt_ave_mms3_dis_numberdensity_fast:.4f} s")
    #     print(f"MMS4: {dt_ave_mms4_dis_numberdensity_fast:.4f} s")
    #     exit()

    for i, mms_id in enumerate(mms_id_list):
        print(f"variation(dt)/average(dt) MMS{mms_id}_dis_numberdensity_fast = {dt_std_norm_mms_dis_ndens_fast[i]:.4f}")


    if dt_std_norm_mms_dis_ndens_fast[0] > 0.001 or \
    dt_std_norm_mms_dis_ndens_fast[1] > 0.001 or \
    dt_std_norm_mms_dis_ndens_fast[2] > 0.001 or \
    dt_std_norm_mms_dis_ndens_fast[3] > 0.001:
        print('skip this interval due to large variations in time steps for dis_numberdensity_fast data.')
        continue
    else:
        print("The time series of dis_numberdensity_fast data have stable time steps.")




    npoint_width1 = int(t_window_de_spin / dt_ave_mms_dis_ndens_fast[0])  # number of points in the window
    tsmooth('mms1_dis_numberdensity_fast', width=npoint_width1, suffix='_smooth')

    npoint_width2 = int(t_window_de_spin / dt_ave_mms_dis_ndens_fast[1])  # number of points in the window
    tsmooth('mms2_dis_numberdensity_fast', width=npoint_width2, suffix='_smooth')

    npoint_width3 = int(t_window_de_spin / dt_ave_mms_dis_ndens_fast[2])  # number of points in the window
    tsmooth('mms3_dis_numberdensity_fast', width=npoint_width3, suffix='_smooth')

    npoint_width4 = int(t_window_de_spin / dt_ave_mms_dis_ndens_fast[3])  # number of points in the window
    tsmooth('mms4_dis_numberdensity_fast', width=npoint_width4, suffix='_smooth')


    # get the smoothed data
    mms1_dis_numberdensity_fast_despin = get_data('mms1_dis_numberdensity_fast_smooth', units=True)
    mms2_dis_numberdensity_fast_despin = get_data('mms2_dis_numberdensity_fast_smooth', units=True)
    mms3_dis_numberdensity_fast_despin = get_data('mms3_dis_numberdensity_fast_smooth', units=True)
    mms4_dis_numberdensity_fast_despin = get_data('mms4_dis_numberdensity_fast_smooth', units=True)





    # plot the dis_numberdensity_fast data for all four MMS probes
    fig, subs = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

    sub = subs[0]
    sub.plot(t_mms_dis_ndens_fast[0], mms_dis_numberdensity_fast[0].y, label='MMS1')
    sub.plot(mms1_dis_numberdensity_fast_despin.times,
            mms1_dis_numberdensity_fast_despin.y, label='MMS1 (smoothed)', linestyle='--')
    sub.set_xlabel('Time (s)')
    sub.set_ylabel('Density (cm$^{-3}$)')
    sub.set_title('MMS1 DIS Number Density Fast')
    sub.legend()


    sub = subs[1]
    sub.plot(t_mms_dis_ndens_fast[1], mms_dis_numberdensity_fast[1].y, label='MMS2')
    sub.plot(mms2_dis_numberdensity_fast_despin.times,
                mms2_dis_numberdensity_fast_despin.y, label='MMS2 (smoothed)', linestyle='--')
    sub.set_xlabel('Time (s)')
    sub.set_ylabel('Density (cm$^{-3}$)')
    sub.set_title('MMS2 DIS Number Density Fast')
    sub.legend()

    sub = subs[2]
    sub.plot(t_mms_dis_ndens_fast[2], mms_dis_numberdensity_fast[2].y, label='MMS3')
    sub.plot(mms3_dis_numberdensity_fast_despin.times,
                mms3_dis_numberdensity_fast_despin.y, label='MMS3 (smoothed)', linestyle='--')
    sub.set_xlabel('Time (s)')
    sub.set_ylabel('Density (cm$^{-3}$)')
    sub.set_title('MMS3 DIS Number Density Fast')
    sub.legend()

    sub = subs[3]
    sub.plot(t_mms_dis_ndens_fast[3], mms_dis_numberdensity_fast[3].y, label='MMS4')
    sub.plot(mms4_dis_numberdensity_fast_despin.times,
                mms4_dis_numberdensity_fast_despin.y, label='MMS4 (smoothed)', linestyle='--')
    sub.set_xlabel('Time (s)')
    sub.set_ylabel('Density (cm$^{-3}$)')  
    sub.set_title('MMS4 DIS Number Density Fast')
    sub.legend()

    fig.suptitle('MMS1-4 DIS Number Density Fast: ' + trange[0] + ' -- ' + trange[1])
    plt.tight_layout()
    # plt.show()
    fig.savefig(dir_figure + trange0_str + '_dis_numberdensity_fast.png')
    plt.close(fig)
    '''

In [ ]:
# # # Fixed by MMS team on Jun 08, 2026
# # Debug -- the 2023-01-18 case is bad because MMS1 data is wrong and has large offsets compared with other three s/c.
# # calculate z+ and z- for each MMS probe 

# tplot([f'mms{mms_id}_dis_numberdensity_fast' for mms_id in mms_id_list])
# tplot([f'mms{mms_id}_dis_bulkv_gse_fast' for mms_id in mms_id_list])



# for mms_id in mms_id_list:
#     print(f"Calculating z+ and z- for MMS{mms_id}......")

#     V = get_data(f'mms{mms_id}_dis_bulkv_gse_fast_interp').y
#     B = get_data(f'mms{mms_id}_fgm_b_gse_srvy_l2_interp').y[:,0:3]

#     t = get_data(f'mms{mms_id}_dis_bulkv_gse_fast_interp').times  # use the time of the bulk velocity data

#     t_range = np.where((t >= time_double(trange[0])) & (t <= time_double(trange[1])))[0]

#     V = V[t_range]
#     B = B[t_range]

#     v1 = np.zeros(V.shape)
#     b1 = np.zeros(B.shape)
#     v1[:,0] = V[:,0] - bulkv_avg[0]  # subtract the average bulk velocity
#     v1[:,1] = V[:,1] - bulkv_avg[1]  # subtract the average bulk velocity
#     v1[:,2] = V[:,2] - bulkv_avg[2]  # subtract the average bulk velocity

#     b1[:,0] = B[:,0] - B_avg[0]  # subtract the average magnetic field
#     b1[:,1] = B[:,1] - B_avg[1]  # subtract the average magnetic field
#     b1[:,2] = B[:,2] - B_avg[2]  # subtract the average magnetic field

#     va1 = b1 * 1e-9 /np.sqrt(ndens_avg * 1e6 * m_p * mu_0) * 1e-3 # km/s


#     print(V[0:20,0])


## Calculate $\nabla \cdot Y^\pm$ from the $Y^\pm$ data for each event

In [ ]:
# Begin analyzing each interval
import glob

# plot the good intervals on the original data to visually check the coverage
dir_output = './output/Yaglom_analysis_all_selected_intervals/'
dir_figure = './figure/Yaglom_analysis_all_selected_intervals/'


# np.savez(file_output, trange = trange, 
#             tau_arr = tau_arr, ndens_avg = ndens_avg, bulkv_avg = bulkv_avg, 
#             B_avg = B_avg, coord_diff_mms_avg = coord_diff_mms_avg, 
#             dX_center = dX_center, Yp_arr = Yp_arr, Ym_arr = Ym_arr)

files = sorted(glob.glob(dir_output + 'Yaglom_law*.npz'))


print(f"Found {len(files)} files in the output directory.")

In [ ]:
import os 
import numpy as np 
import matplotlib.pyplot as plt
from itertools import combinations
from functions import progress_bar

for file_output in files:
    print(f"Loading data from {file_output}...")

    if not os.path.exists(file_output):
        print(f"Error: file {file_output} does not exist. Skipping this file.")
        continue

    
    # if file_output != './output/Yaglom_analysis_all_selected_intervals/Yaglom_law_2023-01-18 09-16-01.407352.npz':
    #     continue

    # get the filename without the directory and extension
    trange0_str = os.path.basename(file_output)# .split('.')[0].split('_')[-1]  # extract the trange0_str from the filename
    ind0 = trange0_str.find('Yaglom_law_') + len('Yaglom_law_')
    ind1 = trange0_str.find('.npz')
    trange0_str = trange0_str[ind0:ind1]  # extract the trange0_str from the filename


    if os.path.exists(dir_output + 'divY_' + trange0_str + '.npz'):
        print(f"divY data for {trange0_str} already exists. Skipping this file.")
        continue

    data = np.load(file_output)

    trange = data['trange']
    tau_arr = data['tau_arr']
    ndens_avg = data['ndens_avg']
    bulkv_avg = data['bulkv_avg']
    B_avg = data['B_avg']
    coord_diff_mms_avg = data['coord_diff_mms_avg']
    dX_center = data['dX_center']
    Yp_arr = data['Yp_arr']
    Ym_arr = data['Ym_arr']


    dX_abs = np.sqrt(np.sum(dX_center**2, axis=1))  # absolute distance-increment of the center of the tetrahedron


    ntau = len(tau_arr)  # number of time increments

    dX_direction = np.zeros((ntau, 3))  # direction of the center of the tetrahedron
    for itau in range(ntau):
        dX_direction[itau, :] = dX_center[itau, :] / np.linalg.norm(dX_center[itau, :])  # direction of the center of the tetrahedron

    dX_direction = dX_direction[0,:]



    tau_start_analysis = 60 #s
    tau_end_analysis = tau_arr[-1] # 3600 #s

    ind_tau_start = np.where(tau_arr >= tau_start_analysis)[0][0]  # index of the start time increment
    ind_tau_end = len(tau_arr) - 1  # index of the end time increment



    # For each dX, we have 4*3+1=13 legs
    # from the 13 legs, we can form C_{13}^4 = 715 tetrahedrons
    nleg = 13
    ncomb = 715  # number of combinations of tetrahedrons
    
    # the leg corresponding to 11 or 22 or 33 or 44 is marked as 00. But we will average (11-44).

    # fisrt, list all possible 13 legs
    leg_list = np.zeros((nleg,2),dtype=int)
    leg_list[0,0] = 0
    leg_list[0,1] = 0
    ind = 1
    for i in range(1,5):
        for j in range(1,5):
            if j == i:
                continue

            leg_list[ind,0] = i 
            leg_list[ind,1] = j
            ind += 1

    # print(leg_list)

    # second, find all combinations of 4 legs from the 13 legs
    tetra_info = np.zeros((ncomb,4),dtype=int) # (tetrahedron index, leg index)
    comb = combinations(range(nleg), 4)  # combinations of 4 legs from the 13 legs
    ind_tetra = 0
    for c in comb:
        for ic in range(4):
            tetra_info[ind_tetra, ic] = c[ic]

        ind_tetra += 1


    # print(tetra_info)

    # exit()

    # third, calculate the relative spatial difference, w.r.t. dX_center, of these legs
    dx_list = np.zeros((nleg,3))
    for ileg in range(1,nleg):
        leg = leg_list[ileg,:]
        if leg[0]==leg[1]:
            print('error: leg[0] == leg[1], should not happen')

        dx_list[ileg,:] = (coord_diff_mms_avg[leg[1]-1,:] - coord_diff_mms_avg[leg[0]-1,:]) 

    # for i in range(nleg):
    #     print(leg_list[i,:], dx_list[i,:])

    # exit()


    dX_subarr = dX_center[ind_tau_start:ind_tau_end+1]
    tau_subarr = tau_arr[ind_tau_start:ind_tau_end+1]
    dX_abs_subarr = dX_abs[ind_tau_start:ind_tau_end+1]  


    divYp_subarr = np.zeros(len(tau_subarr)) # mean of all tetrahedrons
    divYm_subarr = np.zeros(len(tau_subarr)) # mean of all tetrahedrons
    divYp_std_subarr = np.zeros(len(tau_subarr)) # std of all tetrahedrons
    divYm_std_subarr = np.zeros(len(tau_subarr)) # std of the mean of all tetrahedrons

    print('Calculating the divergence of Y+ and Y- for each tau...')
    for itau in range(ind_tau_start, ind_tau_end+1):
        progress_bar(itau-ind_tau_start, ind_tau_end-ind_tau_start+1, 100)
        tau = tau_arr[itau]
        dX0 = dX_center[itau]

        # print('Calculating for tau = {:.1f} s, dX0 = {:.1f} km'.format(tau, np.linalg.norm(dX0)))

        # first, calculate Y+ and Y- for the 13 legs
        Yp = np.zeros((nleg,3))
        Ym = np.zeros((nleg,3))
        for ileg in range(nleg):
            leg = leg_list[ileg,:]

            if leg[0]==0 and leg[1]==0: # leg 00, average 
                for isat in range(Nsat):
                    Yp[ileg,:] = Yp[ileg,:] + Yp_arr[itau,isat,isat,:] 
                    Ym[ileg,:] = Ym[ileg,:] + Ym_arr[itau,isat,isat,:]
                Yp[ileg,:] = Yp[ileg,:] / Nsat
                Ym[ileg,:] = Ym[ileg,:] / Nsat
            else:
                if leg[0] == leg[1]:
                    print('error: leg[0] == leg[1], should not happen')

                Yp[ileg,:] = Yp_arr[itau,leg[0]-1,leg[1]-1,:]
                Ym[ileg,:] = Ym_arr[itau,leg[0]-1,leg[1]-1,:]

        # print(Yp,Ym)
        # exit()


        # second, get the corresponding satellite separation -- not needed
        dX = np.zeros((nleg,3))
        for ileg in range(nleg):
            # dX = dX0 + (r_isat1 - r_isat0)
            dX[ileg,:] = dX0[:] + dx_list[ileg,:]



        # functions used to study why a tetrahedron is invalid
        def if_repeat_pairs(legs):
            n = len(legs)

            for i in range(n-1):
                leg0 = legs[i]

                for j in range(i+1,n):
                    leg1 = legs[j]
                    if leg1[0] == leg0[1] and leg1[1] == leg0[0]:
                        return True 
            return False
        
        def if_two_repeat_pairs(legs):
            n = len(legs)

            n_pair = 0
            for i in range(n-1):
                leg0 = legs[i]

                for j in range(i+1,n):
                    leg1 = legs[j]
                    if leg1[0] == leg0[1] and leg1[1] == leg0[0]:
                        n_pair += 1
            if n_pair == 2:
                return True
            else:
                return False

        def if_has_00(legs):
            for leg in legs:
                if leg[0] == 0 and leg[1] == 0:
                    return True
            return False

        def if_same_tetra(tetra0_legs, tetra1_legs, if_exact=False):
            num_same_legs = 0
            for i in range(4):
                leg1 = tetra1_legs[i]

                for j in range(4):
                    leg0 = tetra0_legs[j]

                    if if_exact:
                        if leg1[0] == leg0[0] and leg1[1] == leg0[1]:
                            num_same_legs += 1
                    else:
                        if (leg1[0] == leg0[1] and leg1[1] == leg0[0]) \
                            or (leg1[0] == leg0[0] and leg1[1] == leg0[1]):
                            num_same_legs += 1

            if num_same_legs == 4:
                return True
            else:
                return False



        # third, take each tetrahedron
        divYp_tetra = []
        divYm_tetra = []

        n_valid_tetra = 0
        n_invalid_tetra = 0
        n_invalid_with_00 = 0
        n_invalid_with_two_pairs = 0
        n_invalid_with_00_and_one_pair = 0
        n_invalid_with_00_and_no_pair = 0 
        n_invalid_tetra_without_repeats = 0
        n_invalid_tetra_without_repeats_and_00 = 0
        n_invalid_tetra_one_pair_no_00 = 0
        for itetra in range(ncomb):
            legs_tetra = tetra_info[itetra,:]  # indices of the 4 legs
            legs_this_tetra = leg_list[legs_tetra,:] # get the legs of this tetrahedron

            dx_vec = dx_list[legs_tetra,:]  # get the dx vector of the legs of this tetrahedron, 4 legs * 3 components


            # determine whether the tetrahedron is valid
            v1 = dx_vec[1,:] - dx_vec[0,:]
            v2 = dx_vec[2,:] - dx_vec[0,:]
            v3 = dx_vec[3,:] - dx_vec[0,:]

            vol_norm = np.linalg.norm(v1) * np.linalg.norm(v2) * np.linalg.norm(v3)
            vol_tetra = np.abs(np.dot(v1, np.cross(v2, v3))) # volume of the tetrahedron

            

            if vol_tetra/vol_norm < 1e-8:
                n_invalid_tetra += 1
                # print('Tetrahedron #{:03d} is invalid. Legs: {}'.format(itetra, leg_list[legs_tetra,:]))

                # see play_with_tetrahedron.py for the details
                if if_has_00(legs_this_tetra):
                    n_invalid_with_00 += 1
                    # print('Tetrahedron #{:03d} is invalid with 00. Legs: {}'.format(itetra, leg_list[legs_tetra,:]))

                if if_two_repeat_pairs(legs_this_tetra):
                    n_invalid_with_two_pairs += 1
                    # print('Tetrahedron #{:03d} is invalid with two repeat pairs. Legs: {}'.format(itetra, leg_list[legs_tetra,:]))

                if if_has_00(legs_this_tetra) and if_repeat_pairs(legs_this_tetra):
                    n_invalid_with_00_and_one_pair += 1
                    # print('Tetrahedron #{:03d} is invalid with 00 and one repeat pair. Legs: {}'.format(itetra, leg_list[legs_tetra,:]))

                if if_has_00(legs_this_tetra) and not if_repeat_pairs(legs_this_tetra):
                    n_invalid_with_00_and_no_pair += 1
                    # print('Tetrahedron #{:03d} is invalid with 00 and no repeat pairs. Legs: {}'.format(itetra, leg_list[legs_tetra,:]))

                if not if_repeat_pairs(legs_this_tetra):
                    n_invalid_tetra_without_repeats += 1
                    # print('Tetrahedron #{:03d} is invalid without repeat pairs. Legs: {}'.format(itetra, leg_list[legs_tetra,:]))

                if if_repeat_pairs(legs_this_tetra) and not if_has_00(legs_this_tetra) and not if_two_repeat_pairs(legs_this_tetra) :
                    n_invalid_tetra_one_pair_no_00 += 1
                    # print('Tetrahedron #{:03d} is invalid with one pair and without 00. Legs: {}'.format(itetra, leg_list[legs_tetra,:]))

                if not if_repeat_pairs(legs_this_tetra) and not if_has_00(legs_this_tetra):
                    n_invalid_tetra_without_repeats_and_00 += 1
                    # print('Tetrahedron #{:03d} is invalid without repeat pairs and without 00. Legs: {}'.format(itetra, leg_list[legs_tetra,:]))
            else:
                # calculate divergence div_l(Y+(l)) and div_l(Y-(l))
                n_valid_tetra += 1

                Yp_vec = Yp[legs_tetra,:]  # get the Y+ vector of the tetrahedron
                divYp = calculate_divF_tetrahedron(dx_vec, Yp_vec)

                Ym_vec = Ym[legs_tetra,:]  # get the Y- vector of the tetrahedron
                divYm = calculate_divF_tetrahedron(dx_vec, Ym_vec)

                divYp_tetra.append(divYp)
                divYm_tetra.append(divYm)


        divYp_tetra = np.array(divYp_tetra)
        divYm_tetra = np.array(divYm_tetra)

        divYp_subarr[itau-ind_tau_start] = np.mean(divYp_tetra)  # mean of all tetrahedrons
        divYm_subarr[itau-ind_tau_start] = np.mean(divYm_tetra)  # mean of all tetrahedrons

        divYp_std_subarr[itau-ind_tau_start] = np.std(divYp_tetra)  # std of all tetrahedrons
        divYm_std_subarr[itau-ind_tau_start] = np.std(divYm_tetra)  # std of all tetrahedrons


        # # plot a histogram of the divergence
        # fig, subs = plt.subplots(2, 1, figsize=(8, 6))
        # sub_p = subs[0]
        # sub_m = subs[1]

        # sub_p.hist(divYp_tetra, bins=20, alpha=0.5, label='divY+')
        # sub_m.hist(divYm_tetra, bins=20, alpha=0.5, label='divY-')

        # sub_p.set_xlabel('divY+')
        # sub_m.set_xlabel('divY-')

        # sub_p.set_ylabel('Count')
        # sub_m.set_ylabel('Count')

        # sub_p.legend()
        # sub_m.legend()
        # plt.tight_layout()
        # plt.show()


        # print('Number of valid tetrahedrons: {}'.format(n_valid_tetra))
        # print('Number of invalid tetrahedrons: {}'.format(n_invalid_tetra))
        # print('Number of invalid tetrahedrons with 00: {}'.format(n_invalid_with_00))
        # print('Number of invalid tetrahedrons with 00 and one repeat pair: {}'.format(n_invalid_with_00_and_one_pair))
        # print('Number of invalid tetrahedrons with 00 and no repeat pairs: {}'.format(n_invalid_with_00_and_no_pair))
        # print('Number of invalid tetrahedrons without repeat pairs: {}'.format(n_invalid_tetra_without_repeats))
        # print('Number of invalid tetrahedrons without repeat pairs and without 00: {}'.format(n_invalid_tetra_without_repeats_and_00))
        # print('Number of invalid tetrahedrons with two repeat pairs: {}'.format(n_invalid_with_two_pairs))
        # print('Number of invalid tetrahedrons with one pair and without 00: {}'.format(n_invalid_tetra_one_pair_no_00))

        # exit()

        
    print('\n')



    # save the divY data to a npz file
    np.savez(dir_output + 'divY_' + trange0_str + '.npz', trange=trange, tau_arr=tau_subarr, dX=dX_subarr, dX_abs=dX_abs_subarr, 
             divYp=divYp_subarr, divYm=divYm_subarr, divYp_std=divYp_std_subarr, divYm_std=divYm_std_subarr)


    # unit of div Y is (km/s)^3 / km = km^2 / s^3 = 10^6 m^2 / s^3 = 10^6 J / (kg s)

    # according to div(Y) = -4 * epsilon, we can calculate epsilon:
    # epsilon_p = -0.25 * divYp
    # epsilon_m = -0.25 * divYm

    epsilon_p_div = -0.25 * divYp_subarr
    epsilon_m_div = -0.25 * divYm_subarr

    epsilon_p_div_std = -0.25 * divYp_std_subarr
    epsilon_m_div_std = -0.25 * divYm_std_subarr



    '''
    # plot in two panels
    fig,subs = plt.subplots(2,1,figsize=(8, 8))

    sub_p = subs[0]
    sub_m = subs[1]

    sub_p.plot(dX_abs_subarr, divYp_subarr, label='divY+')
    # plot a shade 
    low_bound = divYp_subarr - divYp_std_subarr
    high_bound = divYp_subarr + divYp_std_subarr
    sub_p.fill_between(dX_abs_subarr, low_bound, high_bound, alpha=0.2, color='C1')

    sub_m.plot(dX_abs_subarr, divYm_subarr, label='divY-')
    low_bound = divYm_subarr - divYm_std_subarr
    high_bound = divYm_subarr + divYm_std_subarr
    sub_m.fill_between(dX_abs_subarr, low_bound, high_bound, alpha=0.2, color='C2')

    sub_p.set_xlabel(r'$\Delta X$ km')
    sub_m.set_xlabel(r'$\Delta X$ km')
    sub_p.set_ylabel(r'$\nabla_{l} \cdot Y^+$')
    sub_m.set_ylabel(r'$\nabla_{l} \cdot Y^-$')

    plt.tight_layout()

    # fig.savefig('./figure/divYpYm_' + trange0_str + '_shorter_interval.png', dpi=300)

    plt.show()
    plt.close(fig)
    '''



    # # plot Y_{1,1}, Y_{1,2}, Y_{1,3}, Y_{1,4} ------------------------------
    # fig, subs = plt.subplots(2, 1, figsize=(8,6))

    # sub_p = subs[0]
    # sub_m = subs[1]

    # isat0 = 0
    # for isat in range(Nsat):
    #     Yp_vec = Yp_arr[:,isat0, isat, :]
    #     Yp_para = np.dot(Yp_vec, dX_direction)

    #     Ym_vec = Ym_arr[:,isat0, isat, :]
    #     Ym_para = np.dot(Ym_vec, dX_direction)


    #     sub_p.plot(dX_abs, np.abs(Yp_para), label=r'$Y^+_{%1d,%1d}$' % (isat0+1, isat+1)) 
    #     sub_m.plot(dX_abs, np.abs(Ym_para), label=r'$Y^-_{%1d,%1d}$' % (isat0+1, isat+1))

    # sub_p.set_ylabel(r'$Y^+_{\parallel}$')
    # sub_m.set_ylabel(r'$Y^-_{\parallel}$')

    # sub_p.set_yscale('log',base=10)
    # sub_p.set_xscale('log',base=10)
    # sub_m.set_yscale('log',base=10)
    # sub_m.set_xscale('log',base=10)


    # sub_m.set_xlabel(r'$\Delta X$ km')
    # # sub_m.set_xlabel(r'$\tau$ (s)')

    # sub_p.legend()
    # sub_m.legend()
    # plt.tight_layout()
    # plt.show()



    '''
    # plot the self-increment Yp_parallel 
    epsilon_p_para_list = []
    epsilon_m_para_list = []

    fig, subs = plt.subplots(2, 1, figsize=(8,6))

    sub_p = subs[0]
    sub_m = subs[1]


    for isat in range(Nsat):
        # print('MMS', mms_id_list[isat])
        Yp_vec = Yp_arr[:,isat, isat, :]
        Yp_para = np.dot(Yp_vec, dX_direction)

        # print(Yp_para)

        Ym_vec = Ym_arr[:,isat, isat, :]
        Ym_para = np.dot(Ym_vec, dX_direction)

        # according to Y_parallel = -4/3 * l * epsilon, we can calculate epsilon:
        # epsilon = -3/4 * d Y_parallel / d l
        epsilon_p_para = np.zeros(Yp_para.shape)
        epsilon_m_para = np.zeros(Ym_para.shape)

        for itau in range(len(Yp_para)-1):
            epsilon_p_para[itau] = -3/4 * (Yp_para[itau+1] - Yp_para[itau]) / (dX_abs[itau+1] - dX_abs[itau])
        epsilon_p_para[-1] = -3/4 * (Yp_para[-1] - Yp_para[-2]) / (dX_abs[-1] - dX_abs[-2])  # last point

        for itau in range(len(Ym_para)-1):
            epsilon_m_para[itau] = -3/4 * (Ym_para[itau+1] - Ym_para[itau]) / (dX_abs[itau+1] - dX_abs[itau])
        epsilon_m_para[-1] = -3/4 * (Ym_para[-1] - Ym_para[-2]) / (dX_abs[-1] - dX_abs[-2])  # last point

        epsilon_p_para_list.append(epsilon_p_para)
        epsilon_m_para_list.append(epsilon_m_para)


        # plot 
        sub_p.plot(dX_abs, Yp_para, label=f'MMS{mms_id_list[isat]}') # dX_abs
        sub_m.plot(dX_abs, Ym_para, label=f'MMS{mms_id_list[isat]}')

        # sub_p.plot(tau_arr, Yp_para, label=f'MMS{mms_id_list[isat]}') # dX_abs
        # sub_m.plot(tau_arr, Ym_para, label=f'MMS{mms_id_list[isat]}')


        # x_ref = np.linspace(dX_abs[0], dX_abs[-1], 100)
        # coeff = np.polyfit(dX_abs, np.abs(Yp_para), 1)
        # slope = coeff[0]
        # print('Yp_parallel slope:',slope)

        # slope_arr = np.linspace(slope/5,slope*5,5)
        # for islp in range(len(slope_arr)):
        #     y_ref = x_ref * slope_arr[islp] 
        #     sub_p.plot(x_ref, y_ref, linestyle='--', color='k', alpha=0.5)

        # coeff = np.polyfit(dX_abs, np.abs(Ym_para), 1)
        # slope = coeff[0]
        # print('Ym_parallel slope:',slope)

        # slope_arr = np.linspace(slope/5,slope*5,5)
        # for islp in range(len(slope_arr)):
        #     y_ref = x_ref * slope_arr[islp] 
        #     sub_m.plot(x_ref, y_ref, linestyle='--', color='k', alpha=0.5)

    sub_p.set_ylabel(r'$Y^+_{\parallel}$')
    sub_m.set_ylabel(r'$Y^-_{\parallel}$')

    # sub_p.set_yscale('log',base=10)
    # sub_p.set_xscale('log',base=10)
    # sub_m.set_yscale('log',base=10)
    # sub_m.set_xscale('log',base=10)


    sub_m.set_xlabel(r'$\Delta X$ km')
    # sub_m.set_xlabel(r'$\tau$ (s)')

    sub_p.legend()
    sub_m.legend()
    plt.tight_layout()

    #fig.savefig('./figure/YpYm_parallel_' + trange0_str + '_shorter_interval.png', dpi=300)
    plt.show()
    plt.close(fig)
    '''




    # plot the div Y and Y_parallel into one figure
    fig, subs = plt.subplots(2,1, figsize=(8,6))

    sub = subs[0]
    sub.plot(dX_abs_subarr, divYp_subarr, label=r'$ \nabla_{l} \cdot Y^+$')
    # plot a shade 
    low_bound = divYp_subarr - divYp_std_subarr
    high_bound = divYp_subarr + divYp_std_subarr
    sub.fill_between(dX_abs_subarr, low_bound, high_bound, alpha=0.2, color='gray')

    sub.plot(dX_abs_subarr, divYm_subarr, label=r'$ \nabla_{l} \cdot Y^-$')
    low_bound = divYm_subarr - divYm_std_subarr
    high_bound = divYm_subarr + divYm_std_subarr
    sub.fill_between(dX_abs_subarr, low_bound, high_bound, alpha=0.2, color='gray')

    sub.legend()

    sub.set_xlabel(r'$\Delta X$ km')
    sub.set_ylabel(r'$\nabla_{l} \cdot Y^\pm$ km$^2$/s$^3$')



    sub = subs[1]
    epsilon_p_para_list = []
    epsilon_m_para_list = []

    numb_of_grid_used_for_calculate_gradient = 10

    for isat in range(Nsat):
        # print('MMS', mms_id_list[isat])
        Yp_vec = Yp_arr[:,isat, isat, :]
        Yp_para = np.dot(Yp_vec, dX_direction)

        # print(Yp_para)

        Ym_vec = Ym_arr[:,isat, isat, :]
        Ym_para = np.dot(Ym_vec, dX_direction)

        # according to Y_parallel = -4/3 * l * epsilon, we can calculate epsilon:
        # epsilon = -3/4 * d Y_parallel / d l
        epsilon_p_para = np.zeros(Yp_para.shape)
        epsilon_m_para = np.zeros(Ym_para.shape)
 
        # method 1: use only two neighboring points to calculate the gradient, which is less accurate and more noisy
        # for itau in range(len(Yp_para)-1):
        #     epsilon_p_para[itau] = -3/4 * (Yp_para[itau+1] - Yp_para[itau]) / (dX_abs[itau+1] - dX_abs[itau])
        # epsilon_p_para[-1] = -3/4 * (Yp_para[-1] - Yp_para[-2]) / (dX_abs[-1] - dX_abs[-2])  # last point

        # for itau in range(len(Ym_para)-1):
        #     epsilon_m_para[itau] = -3/4 * (Ym_para[itau+1] - Ym_para[itau]) / (dX_abs[itau+1] - dX_abs[itau])
        # epsilon_m_para[-1] = -3/4 * (Ym_para[-1] - Ym_para[-2]) / (dX_abs[-1] - dX_abs[-2])  # last point


        # # method 2: use a central difference method to calculate the gradient, which is more accurate
        # for itau in range(numb_of_grid_used_for_calculate_gradient, len(Yp_para)-numb_of_grid_used_for_calculate_gradient):

        #     epsilon_p_para[itau] = -3/4 * (Yp_para[itau+numb_of_grid_used_for_calculate_gradient] - 
        #                                    Yp_para[itau-numb_of_grid_used_for_calculate_gradient]) / \
        #                                 (dX_abs[itau+numb_of_grid_used_for_calculate_gradient] - 
        #                                  dX_abs[itau-numb_of_grid_used_for_calculate_gradient])
        #     epsilon_m_para[itau] = -3/4 * (Ym_para[itau+numb_of_grid_used_for_calculate_gradient] - 
        #                                    Ym_para[itau-numb_of_grid_used_for_calculate_gradient]) / \
        #                                     (dX_abs[itau+numb_of_grid_used_for_calculate_gradient] - 
        #                                      dX_abs[itau-numb_of_grid_used_for_calculate_gradient])
            

        # method 3: use linear regression to calculate the gradient, which is more accurate and less noisy
        for itau in range(numb_of_grid_used_for_calculate_gradient, len(Yp_para)-numb_of_grid_used_for_calculate_gradient):
            temp_x = dX_abs[itau-numb_of_grid_used_for_calculate_gradient:itau+numb_of_grid_used_for_calculate_gradient+1]
            temp_y_p = Yp_para[itau-numb_of_grid_used_for_calculate_gradient:itau+numb_of_grid_used_for_calculate_gradient+1]
            temp_y_m = Ym_para[itau-numb_of_grid_used_for_calculate_gradient:itau+numb_of_grid_used_for_calculate_gradient+1]

            # try to fit, if the fitting fails, just set as NaN
            try:
                fit_p = np.polyfit(temp_x, temp_y_p, 1)
            except:
                fit_p = [np.nan] * 2

            try:
                fit_m = np.polyfit(temp_x, temp_y_m, 1)
            except:
                fit_m = [np.nan] * 2

            epsilon_p_para[itau] = -3/4 * fit_p[0]  # slope of the linear fit
            epsilon_m_para[itau] = -3/4 * fit_m[0]  # slope of the linear fit


        epsilon_p_para_list.append(epsilon_p_para)
        epsilon_m_para_list.append(epsilon_m_para)


        # plot 
        sub.plot(dX_abs, Yp_para, label=f'MMS{mms_id_list[isat]}',color=f'C{isat}') # dX_abs
        sub.plot(dX_abs, Ym_para ,color=f'C{isat}',linestyle='--')


    sub.set_ylabel(r'$Y^\pm_{\parallel}$ km$^3$/s$^3$')
    # sub.set_yscale('log',base=10)
    # sub.set_xscale('log',base=10)


    sub.set_xlabel(r'$\Delta X$ km')
    leg = sub.legend(loc='upper left')

    sub.add_artist(leg)

    line1, = sub.plot(0,0,color='k',label=r'$Y^+_{\parallel}$')
    line2, = sub.plot(0,0,color='k',linestyle='--',label=r'$Y^-_{\parallel}$')
    sub.legend(handles=[line1, line2], loc='lower right')


    plt.tight_layout()

    fig.savefig(dir_figure + 'divY_and_Y_parallel_' + trange0_str + '.png', dpi=200)
    # plt.show()
    plt.close(fig)






    # plot epsilon calculated using different methods
    epsilon_p_para_avg = np.mean(np.array(epsilon_p_para_list), axis=0)
    epsilon_m_para_avg = np.mean(np.array(epsilon_m_para_list), axis=0)

    fig, subs = plt.subplots(2,1, figsize=(8,6))

    sub = subs[0]

    sub.plot(dX_abs_subarr, epsilon_p_div, label=r'-$\nabla \cdot Y^+$ / 4')
    sub.plot(dX_abs_subarr, epsilon_m_div, label=r'-$\nabla \cdot Y^-$ / 4')

    sub.fill_between(dX_abs_subarr, epsilon_p_div - epsilon_p_div_std, epsilon_p_div + epsilon_p_div_std, alpha=0.2)
    sub.fill_between(dX_abs_subarr, epsilon_m_div - epsilon_m_div_std, epsilon_m_div + epsilon_m_div_std, alpha=0.2)

    sub.set_xlim([dX_abs[0], dX_abs[-1]])
    sub.legend()


    sub = subs[1]
    sub.plot(dX_abs, epsilon_p_para_avg, label=r'-$\frac{3}{4} \frac{d Y^+_{\parallel}}{d \Delta X}$')
    sub.plot(dX_abs, epsilon_m_para_avg, label=r'-$\frac{3}{4} \frac{d Y^-_{\parallel}}{d \Delta X}$')

    sub.set_xlim([dX_abs[0], dX_abs[-1]])

    sub.legend()

    plt.tight_layout()
    fig.savefig(dir_figure + 'epsilon_comparison_' + trange0_str + '.png', dpi=300)
    # plt.show() 
    plt.close(fig)


    # print(Yp_arr.shape)





## Analyze the div(Y) and Y_parallel data

In [55]:
# analysis of the divY data
import glob
import os
import numpy as np
import matplotlib.pyplot as plt


dir_output = './output/Yaglom_analysis_all_selected_intervals/'
dir_figure = './figure/Yaglom_analysis_all_selected_intervals/'

data_files = sorted(glob.glob(dir_output + 'divY_*.npz'))

# print(f"Found {len(data_files)} divY data files in the output directory.")
# print(data_files)

ncase = len(data_files)



mean_divYp_arr = np.zeros(ncase)
mean_divYm_arr = np.zeros(ncase)

fitted_Yp_para_arr = np.zeros(ncase)
fitted_Ym_para_arr = np.zeros(ncase)

var_divYp_arr = np.zeros(ncase)
var_divYm_arr = np.zeros(ncase)

R2_Yp_para_arr = np.zeros(ncase)
R2_Ym_para_arr = np.zeros(ncase)

for ifile, data_file in enumerate(data_files):
    # print(f"Loading divY data from {data_file}...")
    # print(f"Processing file {ifile + 1} of {len(data_files)}")

    if not os.path.exists(data_file):
        print(f"Error: file {data_file} does not exist. Skipping this file.")
        continue

    data = np.load(data_file)

    trange = data['trange']
    tau_arr = data['tau_arr']
    dX = data['dX']
    dX_abs = data['dX_abs']
    divYp = data['divYp']
    divYm = data['divYm']
    divYp_std = data['divYp_std']
    divYm_std = data['divYm_std']

    
    trange0_str = os.path.basename(data_file)# .split('.')[0].split('_')[-1]  # extract the trange0_str from the filename
    ind0 = trange0_str.find('divY_') + len('divY_')
    ind1 = trange0_str.find('.npz')
    trange0_str = trange0_str[ind0:ind1] 

    file_Yaglom = dir_output + 'Yaglom_law_' + trange0_str + '.npz'

    if not os.path.exists(file_Yaglom):
        print(f"Error: file {file_Yaglom} does not exist. Skipping this file.")
        continue


    # continue

    data_Yaglom = np.load(file_Yaglom)

    trange_Yaglom = data_Yaglom['trange']
    tau_arr_Yaglom = data_Yaglom['tau_arr']

    ndens_avg = data_Yaglom['ndens_avg']
    bulkv_avg = data_Yaglom['bulkv_avg']
    B_avg = data_Yaglom['B_avg']
    coord_diff_mms_avg = data_Yaglom['coord_diff_mms_avg']
    coord_diff_mms_rms = data_Yaglom['coord_diff_mms_rms']
    coord_diff_mms_range = data_Yaglom['coord_diff_mms_range']
    dX_center = data_Yaglom['dX_center']
    Yp_arr = data_Yaglom['Yp_arr']
    Ym_arr = data_Yaglom['Ym_arr']


    # if trange is the same
    if not np.array_equal(trange, trange_Yaglom):
        print(f"Error: trange in {data_file} and {file_Yaglom} do not match. Skipping this file.")
        continue

    # if tau_arr and tau_arr_Yaglom are the same, they all start from 60s, and end at min(T/2, 4hr)
    if not np.array_equal(tau_arr, tau_arr_Yaglom):
        print(f"Error: tau_arr in {data_file} and {file_Yaglom} do not match. Skipping this file.")
        continue
    

    # print(ndens_avg, bulkv_avg, B_avg)

    # print(coord_diff_mms_avg.shape) # 4 satellites * 3 components
    # print(coord_diff_mms_rms.shape) # 4 satellites * 1: RMS taken over three components
    # print(coord_diff_mms_range.shape) # 4 satellites * 3 components


    # if dX and dX_center are the same
    if not np.array_equal(dX, dX_center):
        print(f"Error: dX in {data_file} and dX_center in {file_Yaglom} do not match. Skipping this file.")



    dX_direction = np.zeros((len(tau_arr), 3))  # direction of the center of the tetrahedron
    for itau in range(len(tau_arr)):
        dX_direction[itau, :] = dX_center[itau, :] / np.linalg.norm(dX_center[itau, :])  # direction of the center of the tetrahedron

    dX_direction = dX_direction[0,:]

    # print(dX_direction)

    # bulkv_direction = bulkv_avg / np.linalg.norm(bulkv_avg)
    # print('Bulk velocity direction:', bulkv_direction)





    # add some additional analysis here 
    # first, specify range of tau to analyze

    tau_min_ratio = 0.25 # 25% of the maximum tau
    tau_max_ratio = 0.75 # 75% of the maximum tau


    ind_min = int(len(tau_arr) * tau_min_ratio)
    ind_max = int(len(tau_arr) * tau_max_ratio)


    # take the subarray of tau_arr, dX, dX_abs, divYp, divYm, divYp_std, divYm_std
    tau_subarr = tau_arr[ind_min:ind_max+1]
    dX_subarr = dX[ind_min:ind_max+1]
    dX_abs_subarr = dX_abs[ind_min:ind_max+1]

    divYp_subarr = divYp[ind_min:ind_max+1]
    divYm_subarr = divYm[ind_min:ind_max+1]
    divYp_std_subarr = divYp_std[ind_min:ind_max+1]
    divYm_std_subarr = divYm_std[ind_min:ind_max+1]

    Yp_subarr = Yp_arr[ind_min:ind_max+1,:,:,:]
    Ym_subarr = Ym_arr[ind_min:ind_max+1,:,:,:]



    # # test plot of divYp and divYm as a function of tau
    # plt.plot(tau_subarr, divYp_subarr, label=r'$\nabla \cdot Y^+$')
    # plt.plot(tau_subarr, divYm_subarr, label=r'$\nabla \cdot Y^-$')
    # plt.xlabel(r'$\tau$')
    # plt.ylabel(r'$\nabla \cdot Y$')
    # plt.legend()
    # plt.show()

    # # test plot of Yp and Ym as a function of tau for each satellite pair
    # for isat0 in range(4):
    #     plt.plot(tau_subarr, np.dot(Yp_subarr[:,isat0,isat0,:],dX_direction),color='k')

    # plt.plot(tau_subarr, np.dot(Yp_subarr[:,0,1,:],dX_direction), label=r'$Y^+_{1,2}$')
    # plt.plot(tau_subarr, np.dot(Yp_subarr[:,0,2,:],dX_direction), label=r'$Y^+_{1,3}$')
    # plt.plot(tau_subarr, np.dot(Yp_subarr[:,0,3,:],dX_direction), label=r'$Y^+_{1,4}$')
    # plt.plot(tau_subarr, np.dot(Yp_subarr[:,1,2,:],dX_direction), label=r'$Y^+_{2,3}$')
    # plt.plot(tau_subarr, np.dot(Yp_subarr[:,1,3,:],dX_direction), label=r'$Y^+_{2,4}$')
    # plt.plot(tau_subarr, np.dot(Yp_subarr[:,2,3,:],dX_direction), label=r'$Y^+_{3,4}$')
    # plt.xlabel(r'$\tau$')
    # plt.ylabel(r'$Y^+_{\parallel}$')
    # plt.legend()

    # plt.show()



    # evaluate rms(divYp)/mean(divYp) and rms(divYm)/mean(divYm) for the subarray of tau
    mean_divYp = np.mean(divYp_subarr)
    mean_divYm = np.mean(divYm_subarr)

    var_divYp = np.std(divYp_subarr)/np.abs(np.mean(divYp_subarr))
    var_divYm = np.std(divYm_subarr)/np.abs(np.mean(divYm_subarr))


    # evaluate the linearity of Yp_parallell and Ym_parallel with respect to dX_abs, using linear regression
    Yp_para = np.zeros(len(dX_abs))
    Ym_para = np.zeros(len(dX_abs))

    Yp_para_individual = np.zeros((4, len(dX_abs))) # only (11), (22), (33), (44)
    Ym_para_individual = np.zeros((4, len(dX_abs)))

    Yp_para_range = np.zeros((len(dX_abs),2))
    Ym_para_range = np.zeros((len(dX_abs),2))

    for isat in range(4):
        Yp_vec = Yp_arr[:,isat,isat,:]
        Yp_para += np.dot(Yp_vec, dX_direction)
        Yp_para_individual[isat] = np.dot(Yp_vec, dX_direction)

        Ym_vec = Ym_arr[:,isat,isat,:]
        Ym_para += np.dot(Ym_vec, dX_direction)
        Ym_para_individual[isat] = np.dot(Ym_vec, dX_direction)


    Yp_para_range[:,0] = np.min(Yp_para_individual, axis=0)
    Yp_para_range[:,1] = np.max(Yp_para_individual, axis=0)
    Ym_para_range[:,0] = np.min(Ym_para_individual, axis=0)
    Ym_para_range[:,1] = np.max(Ym_para_individual, axis=0)

    Yp_para /= 4
    Ym_para /= 4


    # Yp_para_subarr = np.zeros(len(tau_subarr))
    # Ym_para_subarr = np.zeros(len(tau_subarr))
    # for isat in range(4):
    #     Yp_vec = Yp_subarr[:,isat,isat,:]
    #     Yp_para_subarr += np.dot(Yp_vec, dX_direction)

    #     Ym_vec = Ym_subarr[:,isat,isat,:]
    #     Ym_para_subarr += np.dot(Ym_vec, dX_direction)

    # Yp_para_subarr /= 4
    # Ym_para_subarr /= 4


    Yp_para_general = np.dot(Yp_arr, dX_direction) 
    Ym_para_general = np.dot(Ym_arr, dX_direction) 


    Yp_para_subarr = Yp_para[ind_min:ind_max+1]
    Ym_para_subarr = Ym_para[ind_min:ind_max+1]


    # fit to get dY_para / dX
    try:
        fitted_Yp_para_arr[ifile] = np.polyfit(dX_abs_subarr, Yp_para_subarr, 1)[0]
    except:
        fitted_Yp_para_arr[ifile] = np.nan

    try:
        fitted_Ym_para_arr[ifile] = np.polyfit(dX_abs_subarr, Ym_para_subarr, 1)[0]
    except:
        fitted_Ym_para_arr[ifile] = np.nan




    coeff_Yp = np.polyfit(dX_abs_subarr, Yp_para_subarr, 1)
    coeff_Ym = np.polyfit(dX_abs_subarr, Ym_para_subarr, 1)

    # calculate the linear regression R^2 value
    Yp_para_fit = np.polyval(coeff_Yp, dX_abs_subarr)
    Ym_para_fit = np.polyval(coeff_Ym, dX_abs_subarr)

    # residuals
    ss_res_Yp = np.sum((Yp_para_subarr - Yp_para_fit)**2)
    ss_res_Ym = np.sum((Ym_para_subarr - Ym_para_fit)**2)

    ss_tot_Yp = np.sum((Yp_para_subarr - np.mean(Yp_para_subarr))**2)
    ss_tot_Ym = np.sum((Ym_para_subarr - np.mean(Ym_para_subarr))**2)

    r_squared_Yp = 1 - (ss_res_Yp / ss_tot_Yp)
    r_squared_Ym = 1 - (ss_res_Ym / ss_tot_Ym)


    mean_divYp_arr[ifile] = mean_divYp
    mean_divYm_arr[ifile] = mean_divYm
    var_divYp_arr[ifile] = var_divYp
    var_divYm_arr[ifile] = var_divYm
    R2_Yp_para_arr[ifile] = r_squared_Yp
    R2_Ym_para_arr[ifile] = r_squared_Ym


    # print(r_squared_Yp, r_squared_Ym)



    # plot the figure for individual event



    # two panels, one is div(Yp) and div(Ym)
    # list the parameters for each figure

    fig, subs = plt.subplots(2,1, figsize=(12,14))


    sub = subs[0]
    sub.plot(dX_abs, divYp, label=r'$\nabla_{\mathbf{l}} \cdot \mathbf{Y^+}$')
    # plot the error range
    sub.fill_between(dX_abs, divYp - divYp_std, divYp + divYp_std, alpha=0.2, color='C0')


    # make the two curves with separate y-axis
    sub_t = sub.twinx()
    sub_t.plot(dX_abs, divYm, label=r'$\nabla_{\mathbf{l}} \cdot \mathbf{Y^-}$', color='C1')
    sub_t.fill_between(dX_abs, divYm - divYm_std, divYm + divYm_std, alpha=0.2, color='C1')

    # make the y-axis of sub_t C1 colors
    sub_t.set_ylabel(r'$\nabla_{\mathbf{l}} \cdot \mathbf{Y^-}$   $\mathrm{km}^2 /s^3$', color='C1')
    sub_t.tick_params(axis='y', colors='C1')

    sub_t.spines['right'].set_color('C1')

    sub.axvspan(dX_abs[ind_min], dX_abs[ind_max], alpha=0.3, color='gray')
    sub.set_xlabel(r'$|\Delta \mathbf{x}|$ km')
    sub.set_ylabel(r'$\nabla_{\mathbf{l}} \cdot \mathbf{Y^+}$   $\mathrm{km}^2 /s^3$')
    sub.legend()
    sub_t.legend(loc='lower right')

    sub.set_title(r'$\sigma(\nabla \cdot \mathbf{Y^+})/\mu(\nabla \cdot \mathbf{Y^+})$'+' = ${:.2f}$'.format(var_divYp) 
        + '     ' + 
        r'$\sigma(\nabla \cdot \mathbf{Y^-})/\mu(\nabla \cdot \mathbf{Y^-})$'+' = ${:.2f}$'.format(var_divYm)
        + '\n' + 
        r'$\mu(\nabla \cdot \mathbf{Y^+})$' + ' = ${:.2f}$'.format(mean_divYp)
        + '     ' + 
        r'$\mu(\nabla \cdot \mathbf{Y^-})$' + ' = ${:.2f}$'.format(mean_divYm)
        )




    sub = subs[1]
    sub.plot(dX_abs, Yp_para, label=r'$Y^+_{\parallel}$')
    sub.fill_between(dX_abs, Yp_para_range[:,0], Yp_para_range[:,1], alpha=0.2, color='C0')


    sub_t = sub.twinx()
    sub_t.plot(dX_abs, Ym_para, label=r'$Y^-_{\parallel}$', color='C1')
    sub_t.fill_between(dX_abs, Ym_para_range[:,0], Ym_para_range[:,1], alpha=0.2, color='C1')
    sub_t.set_ylabel(r'$Y^-_{\parallel}$   $\mathrm{km}^2 /s^3$', color='C1')
    sub_t.tick_params(axis='y', colors='C1')

    sub_t.spines['right'].set_color('C1')

    sub.axvspan(dX_abs[ind_min], dX_abs[ind_max], alpha=0.3, color='gray')
    sub.set_xlabel(r'$|\Delta \mathbf{x}|$ km')
    sub.set_ylabel(r'$Y_{\parallel}$   $\mathrm{km}^2 /s^3$')
    sub.legend()

    sub.set_title(r'$R^2(Y_{\parallel}^+)$' + '$= {:.2f}$'.format(r_squared_Yp) + 
        '    ' + r'$R^2(Y_{\parallel}^-)$' + '$= {:.2f}$'.format(r_squared_Ym) + 
        '\n' + 
        r'$\partial Y_{\parallel}^+/ \partial (\Delta X)$' + ' = ${:.4f}$'.format(fitted_Yp_para_arr[ifile]) + 
        '    ' + r'$\partial Y_{\parallel}^-/ \partial (\Delta X)$' + ' = ${:.4f}$'.format(fitted_Ym_para_arr[ifile]))

    # plt.show()


    fig.savefig(dir_figure + 'divY_and_Y_parallel_with_param_' + trange0_str + '.png',dpi=200)
    plt.close(fig)



    # another figure compares Y_{ii} with Y_{ij} -- only for "+"
    # left panel: Y_{11}, Y_{22}, Y_{33}, Y_{44}
    # right panel: Y_{12}, Y_{13}, Y_{14}, Y_{23}, Y_{24}, Y_{34}


    fig, subs = plt.subplots(1,2, figsize=(12,5))

    sub = subs[0]
    sub.plot(dX_abs, Yp_para_general[:,0,0], label=r'$Y^+_{\parallel,11}$')
    sub.plot(dX_abs, Yp_para_general[:,1,1], label=r'$Y^+_{\parallel,22}$')
    sub.plot(dX_abs, Yp_para_general[:,2,2], label=r'$Y^+_{\parallel,33}$')
    sub.plot(dX_abs, Yp_para_general[:,3,3], label=r'$Y^+_{\parallel,44}$')

    sub.set_xlabel(r'$|\Delta \mathbf{x}|$ km')
    sub.set_ylabel(r'$Y_{\parallel,ii}$   $\mathrm{km}^2 /s^3$')
    sub.legend()

    sub = subs[1]
    sub.plot(dX_abs, Yp_para_general[:,0,1], label=r'$Y^+_{\parallel,12}$')
    # sub.plot(dX_abs, Yp_para_general[:,0,2], label=r'$Y^+_{\parallel,13}$')
    # sub.plot(dX_abs, Yp_para_general[:,0,3], label=r'$Y^+_{\parallel,14}$')
    sub.plot(dX_abs, Yp_para_general[:,1,2], label=r'$Y^+_{\parallel,23}$')
    # sub.plot(dX_abs, Yp_para_general[:,1,3], label=r'$Y^+_{\parallel,24}$')
    sub.plot(dX_abs, Yp_para_general[:,2,3], label=r'$Y^+_{\parallel,34}$')

    sub.plot(dX_abs, Yp_para_general[:,1,0], label=r'$Y^+_{\parallel,21}$')
    # sub.plot(dX_abs, Yp_para_general[:,2,0], label=r'$Y^+_{\parallel,31}$')
    # sub.plot(dX_abs, Yp_para_general[:,3,0], label=r'$Y^+_{\parallel,41}$')
    sub.plot(dX_abs, Yp_para_general[:,2,1], label=r'$Y^+_{\parallel,32}$')
    # sub.plot(dX_abs, Yp_para_general[:,3,1], label=r'$Y^+_{\parallel,42}$')
    sub.plot(dX_abs, Yp_para_general[:,3,2], label=r'$Y^+_{\parallel,43}$')

    sub.plot(dX_abs, Yp_para, color='k', linestyle='--', label=r'$\bar{Y}^+_{\parallel,ii}$')
    sub.fill_between(dX_abs, Yp_para_range[:,0], Yp_para_range[:,1], alpha=0.2, color='k')

    # make the two subplot have the same y-range
    y_min = min(subs[0].get_ylim()[0], subs[1].get_ylim()[0])
    y_max = max(subs[0].get_ylim()[1], subs[1].get_ylim()[1])
    subs[0].set_ylim(y_min, y_max)
    subs[1].set_ylim(y_min, y_max)

    sub.set_xlabel(r'$|\Delta \mathbf{x}|$ km')
    sub.set_ylabel(r'$Y_{\parallel,ij}$   $\mathrm{km}^2 /s^3$')
    sub.legend()

    fig.tight_layout()

    # plt.show()

    fig.savefig(dir_figure + 'Y_parallel_ii_and_ij_' + trange0_str + '.png',dpi=200)
    plt.close(fig)
  


In [43]:


# plot the distributions of var_divYp_arr, var_divYm_arr, R2_Yp_para_arr, R2_Ym_para_arr
fig, subs = plt.subplots(1,2, figsize=(12,5))

# plot histogram of Var(divY)
sub = subs[0]

sub.hist(var_divYp_arr, bins=20, range=[0,1], alpha=0.5, edgecolor='black', linewidth=0.5,
    label=r'$\sigma(\nabla_{l} \cdot Y^+)/\mu(\nabla_{l} \cdot Y^+)$')
sub.hist(var_divYm_arr, bins=20, range=[0,1], alpha=0.3, edgecolor='black', linewidth=0.5,
    label=r'$\sigma(\nabla_{l} \cdot Y^-)/\mu(\nabla_{l} \cdot Y^-)$')
sub.set_xlabel(r'$\sigma(\nabla_{l} \cdot Y)/\mu(\nabla_{l} \cdot Y)$')
sub.set_ylabel('Number of Events')
sub.legend()

sub.text(0.02, 0.95, '(a)', transform=sub.transAxes, fontsize=14, va='center')


# plot histogram of R^2 values
sub = subs[1]

sub.hist(R2_Yp_para_arr, bins=20, range=[0,1], alpha=0.5, edgecolor='black', linewidth=0.5,
    label=r'$R^2(Y^+_{\parallel}-\Delta X)$')
sub.hist(R2_Ym_para_arr, bins=20, range=[0,1], alpha=0.3, edgecolor='black', linewidth=0.5,
    label=r'$R^2(Y^-_{\parallel}-\Delta X)$')
sub.set_xlabel(r'$R^2$')
sub.set_ylabel('Number of Events')
sub.legend()
sub.text(0.02, 0.95, '(b)', transform=sub.transAxes, fontsize=14, va='center')
fig.tight_layout()

# plt.show()
fig.savefig('./figure/statistics_vardivY_R2.png',dpi=300)
plt.close(fig)




fig, subs = plt.subplots(1,2, figsize=(12,5))

# statistics of mean_divYp_arr and mean_divYm_arr
sub = subs[0]
sub.hist(mean_divYp_arr, bins=20, range=[-750,750], alpha=0.5, label=r'$\mu(\nabla_{l} \cdot Y^+)$',
        edgecolor='k', color='C0')
sub.axvline(x=np.nanmedian(mean_divYp_arr), color='C0', linestyle='--', linewidth=1)

sub.hist(mean_divYm_arr, bins=20, range=[-750,750], alpha=0.3, label=r'$\mu(\nabla_{l} \cdot Y^-)$', 
        edgecolor='k', color='C1')
sub.axvline(x=np.nanmedian(mean_divYm_arr), color='C1', linestyle='--', linewidth=1)

# sub.axvline(x=0, color='black', linestyle='-', linewidth=1)
sub.set_xlabel(r'$\mu(\nabla_{l} \cdot Y)$ $\mathrm{km}^2 /s^3$')
sub.set_ylabel('Number of Events')
sub.legend()

sub.text(0.02, 0.95, '(a)', transform=sub.transAxes, fontsize=14, va='center')


sub = subs[1]
sub.hist(fitted_Yp_para_arr, bins=20, range=[-0.12,0.12], alpha=0.5, 
    label=r'$\partial Y^+_{\parallel}/\partial (\Delta X)$', edgecolor='k',color='C0')
sub.axvline(x= np.nanmedian(fitted_Yp_para_arr), color='C0', linestyle='--', linewidth=1)


sub.hist(fitted_Ym_para_arr, bins=20, range=[-0.12,0.12], alpha=0.3, 
    label=r'$\partial Y^-_{\parallel}/\partial (\Delta X)$', edgecolor='k',color='C1')
sub.axvline(x= np.nanmedian(fitted_Ym_para_arr), color='C1', linestyle=':', linewidth=1)


# sub.axvline(x=0, color='black', linestyle='-', linewidth=1)
sub.set_xlabel(r'$\partial Y_{\parallel}/\partial (\Delta X)$ $\mathrm{km}^2 /s^3$')
sub.set_ylabel('Number of Events')
sub.legend()

sub.text(0.02, 0.95, '(b)', transform=sub.transAxes, fontsize=14, va='center')
fig.tight_layout()

fig.savefig('./figure/statistics_mean_divY_dYdl.png',dpi=300)
plt.close(fig)




In [31]:
print(np.nanmedian(mean_divYp_arr), np.nanmedian(mean_divYm_arr))
print(np.nanmedian(fitted_Yp_para_arr), np.nanmedian(fitted_Ym_para_arr))
print(np.nanmedian(np.abs(fitted_Yp_para_arr)), np.nanmedian(np.abs(fitted_Ym_para_arr)))


print(np.nanmedian(mean_divYp_arr)/np.nanmedian(np.abs(fitted_Yp_para_arr)), 
np.nanmedian(mean_divYm_arr)/np.nanmedian(np.abs(fitted_Ym_para_arr)))

-89.76021443834397 -2.339051660430754
0.001747213266959903 -0.0005602169037890523
0.02066664005234239 0.009635554480466334
-4343.241775683338 -242.75215974053114


In [ ]:
# Several relatively good cases

# Case 1: 2023-02-18 04-00-03.488012
# Case 2: 2023-03-12 01-51-26.137594
# Case 3: 2023-03-14 11-30-03.773092
# Case 4: 2023-04-01 00-30-02.066913





time_str = '2023-02-18 04-00-03.488012'


# read the exact time intervals from the event list


